In [ ]:
#  CELL 0 : Output capture : Run this FIRST before any other cell 
# All print() output appears in BOTH the cell output AND ps1_outputs/console.log
# All plt.show() figures appear in BOTH the cell output AND ps1_outputs/<name>.png
import os, sys
OUT_DIR = "ps1_validator_oos_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
class _Tee:
    """Writes to Jupyter cell output + log file simultaneously."""
    def __init__(self, jupyter_stream, log_fh):
        self._jupyter = jupyter_stream   # Jupyter OutStream -> shows in cell
        self._log     = log_fh           # file handle       -> saved to disk
        self._is_tee  = True
    def write(self, data):
        try:
            self._jupyter.write(data)
            self._jupyter.flush()
        except Exception:
            pass
        try:
            self._log.write(data)
            self._log.flush()
        except Exception:
            pass
    def flush(self):
        try: self._jupyter.flush()
        except Exception: pass
        try: self._log.flush()
        except Exception: pass
    def __getattr__(self, name):
        return getattr(self._jupyter, name)
if not getattr(sys.stdout, "_is_tee", False):
    _log_fh    = open(os.path.join(OUT_DIR, "console.log"), "w", buffering=1, encoding="utf-8")
    sys.stdout = _Tee(sys.stdout, _log_fh)
    sys.stderr = _Tee(sys.stderr, _log_fh)
    print("[capture] All output -> cell output + " + os.path.join(OUT_DIR, "console.log"))
else:
    print("[capture] Already active.")
try:
    import matplotlib.pyplot as _plt
    if not getattr(_plt.show, "_wrapped", False):
        _orig_show = _plt.show
        # Counter stored as mutable default arg on the function itself --
        # immune to any variable-name collision from other cells in the notebook.
        def _show(*a, _cnt=[0], **k):
            # The capture/save-to-disk step below is a nice-to-have side effect,
            # not the actual render -- it must never be able to block _orig_show()
            # from running. Previously accessed _fig._suptitle directly (a private
            # matplotlib attribute, not the public get_suptitle() API); if that
            # attribute's shape ever changes across a matplotlib version bump (the
            # same class of break seen with sklearn's _find_floating_dtype_allow_sparse
            # this session), the resulting exception happened BEFORE _orig_show(*a, **k)
            # at the end, so the plot never actually rendered -- looked exactly like
            # "matplotlib doesn't render" on every single plt.show() call.
            try:
                for _fig_num in _plt.get_fignums():
                    try:
                        _fig = _plt.figure(_fig_num)
                        _suptitle_text = _fig.get_suptitle().strip() if hasattr(_fig, "get_suptitle") else ""
                        _title = (
                            _suptitle_text
                            or (_fig.axes[0].get_title().strip() if _fig.axes else "")
                            or f"fig_{_cnt[0]:02d}"
                        )
                        _safe = "".join(
                            c if c.isalnum() or c in "-_ " else "_" for c in _title
                        ).strip().replace(" ", "_")[:60]
                        _fname = os.path.join(OUT_DIR, f"{_safe}_{_cnt[0]:02d}.png")
                        _fig.savefig(_fname, dpi=110, bbox_inches="tight")
                        print(f"[capture] fig saved -> {_fname}")
                        _cnt[0] += 1
                    except Exception as _save_err:
                        print(f"[capture] fig save failed (fig {_fig_num}): {_save_err}")
            except Exception as _capture_loop_err:
                print(f"[capture] figure capture skipped entirely: {_capture_loop_err}")
            return _orig_show(*a, **k)
        _show._wrapped = True
        _plt.show = _show
    print(f"[capture] matplotlib patched -> figures saved to {OUT_DIR}/")
except Exception as _cap_err:
    print("[capture] matplotlib not ready:", _cap_err)

In [ ]:
# =============================================================================
#  CELL 0B : Run profile -- the ONE place to choose this run's configuration
# =============================================================================
# PROFILE = "none" leaves the environment exactly as it is, so a default run is
# unchanged. Any other profile sets every PS1_* variable the run needs, removes any
# stale PS1_* setting left in this kernel, and touches nothing else. It must run
# before CELL 4 (the checkpoint signature is read there) and CELL 5 (Spark starts
# the JVM), which is why it sits here.
#
# FRESH_REBUILD = True rebuilds ETL and training from scratch.
# FRESH_REBUILD = False resumes from the last checkpoint -- use it after a restart
# or a crash once training has finished, so the models are not rebuilt.
import os

PROFILE = "none"          # "none" | "baseline_29aug" | "baseline_29aug_v2"
FRESH_REBUILD = True

PROFILES = {
    # The 29-Aug-2026 data baseline, decided 24-Sep-2026: the guard-run configuration
    # read back from its checkpoint, plus labelling only what is observable. Cubic's
    # EDW.DEVICE_EVENT froze at 2026-08-29, so at a 7-day horizon labels end 2026-08-22.
    "baseline_29aug": {
        "PS1_EVENT_DEFINITION": "ventra_kpi",
        "PS1_LABEL_HORIZON_DAYS": "7",
        "PS1_EXCLUDE_RELIEVED": "true",
        "PS1_EVENT_SESSION_GAP_DAYS": "3",
        "PS1_LABEL_OBSERVED_EDGE": "true",
        "PS1_COMPLETENESS_GUARD": "true",
        "PS1_COVERAGE_SIGNAL": "fleet",
        "PS1_COVERAGE_FLOOR": "0.5",
        "PS1_COVERAGE_LOOKBACK_DAYS": "30",
        "PS1_COVERAGE_CONTROL_MONTH": "2025-05",
        "PS1_ENABLE_AVAIL_FEATURES": "true",
        "PS1_ENABLE_EVENT_QUALITY_FEATURES": "true",
        "PS1_ENABLE_M401_FEATURES": "true",
        "PS1_ENABLE_USAGE_EXT_FEATURES": "true",
        "PS1_ENABLE_WARNING_FEATURES": "true",
        "PS1_EVQ_PRIOR_ONLY": "true",
        "PS1_EXCLUDE_GAP_FEATURES": "true",
    },
}

# 25-Sep v2: the baseline, scored only on in-service device-days, with lookaheads that
# cross an interior feed gap masked, and without the features whose day-D value includes
# day D or later.
PROFILES["baseline_29aug_v2"] = {**PROFILES["baseline_29aug"],
                                 "PS1_EXCLUDE_ACTIVE_EPISODE": "true",
                                 "PS1_LABEL_MASK_UNOBSERVED": "true",
                                 "PS1_EXCLUDE_SAME_DAY_FEATURES": "true",
                                 # operate at the recall floor (GATE/TVM 0.50, VALIDATOR 0.80)
                                 "PS1_THRESHOLD_RULE": "recall_floor"}

if PROFILE == "none":
    print("[profile] none -- environment left exactly as it is")
else:
    if PROFILE not in PROFILES:
        raise ValueError(f"unknown PROFILE {PROFILE!r}; choose from {['none', *PROFILES]}")
    _want = PROFILES[PROFILE]
    _stale = sorted(k for k in os.environ
                    if k.startswith("PS1_") and k not in _want and k != "PS1_FORCE_RELOAD")
    for _k in _stale:
        del os.environ[_k]
    os.environ.update(_want)
    os.environ["PS1_FORCE_RELOAD"] = "true" if FRESH_REBUILD else "false"
    # A larger JIT code cache. With the default the compiler switched itself off
    # mid-training on 24-Sep and every later stage ran interpreted.
    _jto = os.environ.get("JAVA_TOOL_OPTIONS", "")
    if "ReservedCodeCacheSize" not in _jto:
        os.environ["JAVA_TOOL_OPTIONS"] = (_jto + " -XX:ReservedCodeCacheSize=512m").strip()
    print(f"[profile] {PROFILE}: {len(_want)} settings applied")
    for _k in sorted(_want):
        print(f"[profile]   {_k} = {_want[_k]}")
    if _stale:
        print(f"[profile] removed stale settings left in this kernel: {_stale}")
    print(f"[profile] FRESH_REBUILD={FRESH_REBUILD} -> PS1_FORCE_RELOAD={os.environ['PS1_FORCE_RELOAD']}")


## PS1 — VALIDATOR 3-day Failure Prediction

**Scope:** VALIDATOR (BMV) devices only (`mars_device_category = 'VALIDATOR'`)

**Label (R6-1 2026-07-17):**  
`will_fail_3d = 1` when a hardware OOS Set event exists in `silver.device_failures` (source: `device_event_enriched` where `is_oos_event=TRUE AND EVENT_STATE_TYPE_NAME='Set' AND mars_device_category='VALIDATOR'`) within the next 3 calendar days. Label is pre-computed in `device_ps1_daily__create.sql` — no notebook-level label derivation needed.

Label is now built in Gold SQL (`device_ps1_daily__create.sql`) via the `outage_label_days_3d` CTE using a `device_failures` UNION — the same CTEs as TVM/GATE but sourced from `device_failures` where `device_category = 'VALIDATOR'` instead of availability_events.

> **Changed from R5-1 2026-07-16** (deprecated): previously `will_fail_3d = 1` when the device's tap rejection rate exceeded its rolling 28-day mean + 2 SD in the next 3 days. That tap-anomaly label is replaced by the hardware OOS Set event label above.

**Pre-requisite:** run `run_layer_silver` (builds S26 `device_failures`) then `run_layer_gold` (builds `device_ps1_daily`) before running this notebook.  
Then verify: `SELECT round(100.0*avg(will_fail_3d),3) FROM mars_dev.gold.device_ps1_daily WHERE mars_device_category='VALIDATOR'` should be > 0.

**Weather:** Open-Meteo historical API (free, no key) — VALIDATOR bus devices are matched to facility lat/lon via FACILITY_ID from `dim_device`.

**Checkpoint dir:** `ps1_validator_oos_outputs/checkpoints`  
**MLflow experiment:** `chicago-ps1v-validator-failure`
> **Architecture:** VALIDATOR uses **PySpark + Spark ML** (same pipeline as GATE/TVM). Prior pandas/sklearn + SMOTE path replaced.


**PS1 — Device Failure Prediction (3-day Binary) — VALIDATOR**

---
SageMaker Studio | MLflow | Feature Store | Model Registry | Model Monitor

Architecture : CUBIC MARS AWS sparkml-v1
  Gold/Silver S3 (PySpark) → Spark ML distributed training → Feature Store (offline+online)
  → MLflow Experiment (mlflow.spark) → SageMaker Model Registry → Endpoint deploy
  → SageMaker Endpoint → SageMaker Model Monitor (data quality + model quality)

Target  : will_fail_3d  (0=no failure within 3 days, 1=failure within 3 days)
Scope   : VALIDATOR only (hardware OOS Set event label, R6-1 2026-07-17)
Grain   : one row per DEVICE_ID per day
Run in  : SageMaker Studio JupyterLab  (kernel: Python 3 — Data Science 3.0)

---


In [ ]:
#  CELL 1 : Install dependencies 
import subprocess, sys
pkgs = [
    "s3fs", "pyarrow",
    "scikit-learn>=1.4",
    "xgboost==2.1.3", "lightgbm>=4.3", "catboost>=1.2",
    "optuna>=3.6", "shap>=0.45",
    "mlflow>=2.13", "sagemaker-mlflow",
    "boto3", "sagemaker>=2.220,<3.0",
    "pandas>=2.0", "matplotlib", "seaborn",
    "imbalanced-learn",
    "pyspark>=3.4",
]
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
except subprocess.CalledProcessError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps"] + pkgs)
print("All packages installed.")

# Spark boosters + Optuna + SHAP
for _pkg in ["xgboost==2.1.3", "optuna>=3.0", "shap>=0.44"]:
    try:
        _mod = _pkg.split("[")[0].split(">=")[0].split("==")[0]
        __import__(_mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])
try:
    import synapse.ml  # noqa: F401 — requires SynapseML JAR in CELL 5
except ImportError:
    print("[install] synapseml pip optional — LightGBM uses spark.jars.packages SynapseML JAR")


In [ ]:
#  CELL 2 : Imports
import os, json, time, warnings
import math
import platform
from contextlib import contextmanager

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
from pyspark.ml import Pipeline as SparkPipeline
from pyspark.ml.feature import Imputer, VectorAssembler
from pyspark.ml.classification import (
    LogisticRegression as SparkLogisticRegression,
    RandomForestClassifier as SparkRandomForestClassifier,
    GBTClassifier as SparkGBTClassifier,
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap, optuna, boto3, sagemaker, joblib, tarfile

# deltalake removed: tables exported as Parquet (DeltaProtocolError on deletionVectors)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_selection import mutual_info_classif
from sklearn.isotonic import IsotonicRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import (
    roc_auc_score, roc_curve, f1_score, precision_score, recall_score,
    average_precision_score, precision_recall_curve,
    confusion_matrix, classification_report,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from catboost import CatBoostClassifier

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm
import mlflow.spark
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient

try:
    from sagemaker import Session
except ImportError:
    from sagemaker.session import Session
try:
    from sagemaker import get_execution_role
except ImportError:
    from sagemaker.session import get_execution_role
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker.feature_store.feature_definition import FeatureDefinition, FeatureTypeEnum

STAGE_TIMINGS_SEC = {}

@contextmanager
def timed_stage(name):
    started = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - started
        STAGE_TIMINGS_SEC[name] = round(elapsed, 2)
        print(f"[{name}] {elapsed:,.1f} sec")

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", 60)
print("Imports OK.")


In [ ]:
import os
#  CELL 3 : Configuration
# Fill in your artifact bucket name (the dedicated artefacts bucket)
ARTIFACT_BUCKET = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
PS1_XW_S3_PREFIX = "chicago/device_ps1_cross_wired_daily"
PS1_XW_S3 = f"s3://{ARTIFACT_BUCKET}/{PS1_XW_S3_PREFIX}"

GOLD_BUCKET = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX = "chicago/gold/device_ps1_daily"
GOLD_S3 = f"s3://{GOLD_BUCKET}/{GOLD_PREFIX}"

SM_SESSION = Session()
ROLE = get_execution_role()
REGION = boto3.Session().region_name
SM_CLIENT = boto3.client("sagemaker", region_name=REGION)
S3_CLIENT = boto3.client("s3", region_name=REGION)

# MLflow — SageMaker Managed Tracking Server
MLFLOW_SERVER_ARN = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN

try:
    MLFLOW_TRACKING_URI = SM_CLIENT.describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev"
    )["TrackingServerUrl"]
    print(f"MLflow managed server: {MLFLOW_TRACKING_URI}")
except Exception:
    MLFLOW_TRACKING_URI = "file:///home/sagemaker-user/mlruns"
    print("MLflow managed server not found — using local file fallback (EFS)")

# SageMaker managed MLflow requires the ARN (HTTPS URL alone → "Missing Tracking Server ARN").
mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)

GOLD_VERSION = 6

# Optuna HPT checkpoint (joblib) — reused across kernel restarts
# Colocated with CELL 0 outputs: ps1_validator_oos_outputs/checkpoints/
if "OUT_DIR" not in globals():
    OUT_DIR = "ps1_validator_oos_outputs"
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints")
CHECKPOINT_FILE = os.path.join(CKPT_DIR, "optuna_params_oos_3d.joblib")
os.makedirs(CKPT_DIR, exist_ok=True)

# Feature Store
ENABLE_FEATURE_STORE = True
FS_GROUP_NAME = "chicago-ps1-3d-validator-failure-features"

TARGET_HORIZON = "3d"

# OOS label (hardware OOS Set — built in CELL 6 from silver.device_event_enriched)
TARGET_COL = "will_hardware_oos_3d"
SLA_TARGET_COL = "will_fail_3d"  # chargeable SLA reference only
LABEL_HORIZON_DAYS = int(os.environ.get("PS1_LABEL_HORIZON_DAYS", "3"))
DEVICE_CAT = "VALIDATOR"

# Canonical SageMaker / MLflow name suffix (see ps1_resource_names in CELL 4)
PS1_NAME_SUFFIX = "failure"

# Planned maintenance event types (VALIDATOR/BMV) — documented for SLA audits;
# OOS training target is will_hardware_oos_3d built in CELL 6 (same as GATE/TVM).
VALIDATOR_PLANNED_EVENT_TYPES = [105, 106, 108, 129, 135, 140, 507, 516, 522]

# PySpark ETL (ported from OOS Optimized v3)
S3_SILVER = f"s3://{GOLD_BUCKET}/chicago/silver"
S3_GOLD = f"s3://{GOLD_BUCKET}/chicago/gold"
TRAIN_START = "2023-07-01"
MAX_LOOKBACK_DAYS = 90
FEATURE_HISTORY_START = (
    pd.Timestamp(TRAIN_START) - pd.Timedelta(days=MAX_LOOKBACK_DAYS)
).date().isoformat()
LABEL_DATA_LAG_DAYS = 0
PS5_DATE_COLUMN = None
CACHE_FEATURE_FRAME = True
RUN_DEEP_GRAIN_AUDIT = False
AUXILIARY_DUPLICATE_POLICY = "skip"
MAX_DRIVER_MEMORY_FRACTION = 0.55
PANDAS_MEMORY_OVERHEAD_FACTOR = 2.2
ALLOW_OVERSIZED_COLLECT = False
LOCAL_SPARK_TEMP_DIR = "/tmp/ps1_validator_oos_mlflow_spark"
HADOOP_AWS_PACKAGE = "org.apache.hadoop:hadoop-aws:3.3.4"
PYSPARK_VERSION = "3.5.1"
AUTO_INSTALL_LOCAL_JAVA = True
OPENJDK_VERSION = "17"

# Spark booster JARs (SynapseML LightGBM + CatBoost-Spark) — used in CELL 5
SYNAPSEML_PACKAGE = "com.microsoft.azure:synapseml_2.12:1.1.3"
SYNAPSEML_MAVEN_REPO = "https://mmlspark.azureedge.net/maven"
SPARK_JARS_EXCLUDES = (
    "org.scala-lang:scala-reflect,org.apache.spark:spark-tags_2.12,"
    "org.scalactic:scalactic_2.12,org.scalatest:scalatest_2.12,"
    "com.fasterxml.jackson.core:jackson-databind"
)
CATBOOST_SPARK_PACKAGE = "ai.catboost:catboost-spark_3.5_2.12:1.2.8"
SPARK_JARS_PACKAGES = ",".join([HADOOP_AWS_PACKAGE, SYNAPSEML_PACKAGE, CATBOOST_SPARK_PACKAGE])
USE_SPARK_ML = True

# Local Spark booster caps (SageMaker Studio JVM OOM guard)
SPARK_LOCAL_MAX_WORKERS = 2
SPARK_LOCAL_BOOSTER_ROUNDS = 300
SPARK_CLUSTER_BOOSTER_ROUNDS = 800

# SynapseML LGB / CatBoost-Spark native libs SIGSEGV on local[*] Spark — skip on SageMaker by default.
SPARK_LOCAL_SKIP_SYNAPSE_LGB = False
SPARK_LOCAL_SKIP_SYNAPSE_CATBOOST = False
SPARK_LGB_KWARGS = {
    "useSingleDatasetMode": True,
    "dataTransferMode": "bulk",
    "numThreads": 1,
    "passThroughArgs": "force_col_wise=true",
}



def _detected_memory_limit_gb():
    candidates = []
    try:
        import psutil
        candidates.append(psutil.virtual_memory().total)
    except Exception:
        if hasattr(os, "sysconf") and "SC_PHYS_PAGES" in os.sysconf_names:
            candidates.append(os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE"))
    try:
        value = open("/sys/fs/cgroup/memory.max", encoding="utf-8").read().strip()
        if value != "max" and 0 < int(value) < (1 << 60):
            candidates.append(int(value))
    except (FileNotFoundError, OSError, ValueError):
        pass
    try:
        value = int(open("/sys/fs/cgroup/memory/memory.limit_in_bytes", encoding="utf-8").read())
        if 0 < value < (1 << 60):
            candidates.append(value)
    except (FileNotFoundError, OSError, ValueError):
        pass
    return min(candidates) / (1024 ** 3) if candidates else 16.0


DETECTED_MEMORY_GB = _detected_memory_limit_gb()
LOCAL_SPARK_DRIVER_MEMORY_GB = max(1, min(12, int(DETECTED_MEMORY_GB * 0.35)))
LOCAL_SPARK_DRIVER_MEMORY = f"{LOCAL_SPARK_DRIVER_MEMORY_GB}g"
LOCAL_SPARK_THREADS = max(
    1, min(8, os.cpu_count() or 2, max(1, int(DETECTED_MEMORY_GB // 4)))
)
LOCAL_SHUFFLE_PARTITIONS = max(64, min(512, LOCAL_SPARK_THREADS * 32))

print(f"Region               : {REGION}")
print(f"Gold S3              : {GOLD_S3}")
print(f"Silver S3            : {S3_SILVER}")
print(f"Artifacts            : s3://{ARTIFACT_BUCKET}/")
print(f"PS1 cross-wired S3   : {PS1_XW_S3}/{{gate|tvm|validator}}/")
print(f"MLflow URI           : {MLFLOW_TRACKING_URI}")
print(f"Enable Feature Store : {ENABLE_FEATURE_STORE}  ({FS_GROUP_NAME})")
print(f"Target               : {TARGET_COL}  (ref SLA: {SLA_TARGET_COL})")
print(f"Spark JARs           : {SPARK_JARS_PACKAGES[:80]}...")

# Evaluation Fix: operational constants
DISPATCH_BUDGET_PCT = 0.25
SMOTE_RATIO = 0.10
OVERSAMPLE_RATIO = 0.05
SPW_CAP = 25
SPW_CANDIDATES = [10, 15, 20, 25, 30]
NULL_DROP_THRESH = 0.90
ZERO_DROP_THRESH = 0.95
USE_MI_PRUNING = True
MI_TOP_N = 100
TAP_PROTECT_PAT = ("tap_reject", "tap_volume", "peak_hour_tap", "tap_event", "reject_rate")
LEAKAGE_AUC_TH = 0.95
QUALITY_RECALL_FLOOR = {"VALIDATOR": 0.80}
TVM_PLANNED_EVENT_TYPES = [
    105, 106, 108, 129, 135, 140, 184, 407, 507, 516, 521, 522, 523, 1406,
]
GATE_PLANNED_EVENT_TYPES = [105, 106, 108, 129, 135, 140, 151]
CHAMPION_AP_TOL = 0.002

print(f"DISPATCH_BUDGET_PCT  : {DISPATCH_BUDGET_PCT:.0%}")
print(f"SMOTE_RATIO          : {SMOTE_RATIO}")
print(f"SPW_CAP              : {SPW_CAP}")
print(f"OVERSAMPLE_RATIO     : {OVERSAMPLE_RATIO}")
print(f"MI_TOP_N             : {MI_TOP_N}")
print(f"QUALITY_RECALL_FLOOR : {QUALITY_RECALL_FLOOR}")

BUCKET = GOLD_BUCKET
SILVER_PREFIX = "chicago/silver"
GOLD_BASE = "chicago/gold"
S3_OPTS = {"client_kwargs": {"region_name": "us-east-1"}}


In [ ]:
#  CELL 4 : Spark checkpoint — load/save (skip Cells 6-12 when restored)
# Saves on disk under OUT_DIR/checkpoints (see CELL 3 CHECKPOINT_FILE):
#   spark_ckpt_meta.joblib  — FEATURE_COLS, CATEGORIES meta, stage, champion info
#   spark_splits/           — parquet df_ml_spark (train/val/test via split column)
#   spark_models/{name}/    — PipelineModel per SPARK_ML_RESULTS entry
#   spark_eval/test_pdf.parquet — threshold/QG outputs (after CELL 13)
#
# Stages: etl → trained → evaluated
# Usage: Cells 1-3 → this cell → CELL 5 (Spark session + restore) → jump by stage:
#   etl       → CELL 10 (EDA) or CELL 11 (train)
#   trained   → CELL 13 (threshold) or CELL 14+
#   evaluated → CELL 14+ (Feature Store / registration / deploy)
import os, shutil, joblib

CKPT_DIR = os.path.dirname(CHECKPOINT_FILE)
SPARK_CKPT_META = os.path.join(CKPT_DIR, "spark_ckpt_meta.joblib")
SPARK_CKPT_SPLITS = os.path.join(CKPT_DIR, "spark_splits")
SPARK_CKPT_MODELS = os.path.join(CKPT_DIR, "spark_models")
SPARK_CKPT_EVAL = os.path.join(CKPT_DIR, "spark_eval")

# PS1_FORCE_RELOAD=true ignores any on-disk checkpoint and re-runs ETL + training.
# Required whenever the label configuration changed, because a checkpoint restored
# at stage "evaluated" skips CELLS 6-13 entirely and replays the previous label.
FORCE_RELOAD = os.environ.get("PS1_FORCE_RELOAD", "false").strip().lower() == "true"
_PS1_LABEL_NON_DEFAULT = (
    os.environ.get("PS1_EVENT_DEFINITION", "hardware_oos").strip().lower() != "hardware_oos"
    or os.environ.get("PS1_LABEL_HORIZON_DAYS", "3").strip() != "3"
    or os.environ.get("PS1_EXCLUDE_RELIEVED", "false").strip().lower() == "true"
    # Sessionisation changes the label as much as the event definition does, and it is
    # the switch the completeness guard rides on. Its absence here meant a sessionised
    # run could both replay a checkpoint AND pass the publish guards in the cells below.
    or os.environ.get("PS1_EVENT_SESSION_GAP_DAYS", "0").strip() != "0"
    or os.environ.get("PS1_LABEL_MODE", "state").strip().lower() != "state"
    # Dropping the unobservable right edge changes which rows carry a label at all.
    or os.environ.get("PS1_LABEL_OBSERVED_EDGE", "false").strip().lower() == "true"
    # Both change which rows carry a label at all.
    or os.environ.get("PS1_EXCLUDE_ACTIVE_EPISODE", "false").strip().lower() == "true"
    or os.environ.get("PS1_LABEL_MASK_UNOBSERVED", "false").strip().lower() == "true"
)
_PS1_FEATURE_FLAGS = (
    "PS1_ENABLE_WARNING_FEATURES", "PS1_ENABLE_AVAIL_FEATURES",
    "PS1_ENABLE_M401_FEATURES", "PS1_ENABLE_USAGE_EXT_FEATURES",
    "PS1_ENABLE_EVENT_QUALITY_FEATURES",
    # These three change the feature matrix and were blind to the gate as well.
    "PS1_EXCLUDE_GAP_FEATURES", "PS1_EVQ_PRIOR_ONLY", "PS1_ENABLE_KPI_FEATURES",
    "PS1_EXCLUDE_FAILURE_HISTORY",
    # Two silver sources ported from the PS3 v2 engine. Unregistered, a feature
    # flag is invisible to BOTH gates: the checkpoint signature would certify a
    # replay built under a different feature matrix, and all three publish fences
    # would stay open on an experimental run.
    "PS1_ENABLE_STATION_FEATURES", "PS1_ENABLE_UPTIME_FEATURES",
    "PS1_EXCLUDE_SAME_DAY_FEATURES",
)
_PS1_FEATURES_NON_DEFAULT = any(
    os.environ.get(_f, "false").strip().lower() == "true" for _f in _PS1_FEATURE_FLAGS
)
# Every env var that changes the LABEL or the FEATURE MATRIX, stamped into the
# checkpoint so a later run can tell what it is about to replay. TARGET_COL is the
# literal "will_hardware_oos_3d" under every configuration, so without this the meta
# cannot distinguish an experimental checkpoint from a production one.
_PS1_SIGNATURE_VARS = (
    "PS1_EVENT_DEFINITION", "PS1_LABEL_HORIZON_DAYS", "PS1_EXCLUDE_RELIEVED",
    "PS1_EVENT_SESSION_GAP_DAYS", "PS1_LABEL_MODE", "PS1_COMPLETENESS_GUARD",
    "PS1_COVERAGE_SIGNAL", "PS1_COVERAGE_FLOOR", "PS1_COVERAGE_LOOKBACK_DAYS",
    "PS1_LABEL_OBSERVED_EDGE", "PS1_EXCLUDE_ACTIVE_EPISODE", "PS1_LABEL_MASK_UNOBSERVED",
) + _PS1_FEATURE_FLAGS
_PS1_LABEL_SIGNATURE = {
    _k: os.environ.get(_k, "").strip().lower() for _k in _PS1_SIGNATURE_VARS
}
_PS1_CONFIG_NON_DEFAULT = _PS1_LABEL_NON_DEFAULT or _PS1_FEATURES_NON_DEFAULT
if _PS1_CONFIG_NON_DEFAULT and not FORCE_RELOAD:
    print("[CHECKPOINT] !! Configuration is NON-DEFAULT but PS1_FORCE_RELOAD is not "
          "set. A checkpoint on disk was built with a DIFFERENT configuration and "
          "would be replayed instead of rebuilt. Set PS1_FORCE_RELOAD=true.")
CHECKPOINT_LOADED = False
CHECKPOINT_STAGE = None
SPARK_CKPT_VERSION = 2


def _serializable_categories(categories):
    return {
        cat: {k: v for k, v in meta.items() if not k.endswith("_sdf")}
        for cat, meta in categories.items()
    }


def _serializable_category_results(category_results):
    """Strip Spark/non-pickleable objects before joblib checkpoint write."""
    out = {}
    for cat, blob in category_results.items():
        row = {}
        for kk, vv in blob.items():
            if kk in (
                "champion_model", "test_pdf", "predictions",
                "val_pred", "test_pred", "champion_run",
            ):
                continue
            row[kk] = vv
        if "champion_run_id" not in row:
            _cr = blob.get("champion_run")
            if _cr is not None:
                _rid = getattr(getattr(_cr, "info", None), "run_id", None)
                if _rid:
                    row["champion_run_id"] = _rid
        out[cat] = row
    return out


def _infer_ckpt_stage_from_disk():
    if os.path.exists(os.path.join(SPARK_CKPT_EVAL, "test_pdf.parquet")):
        return "evaluated"
    if os.path.isdir(SPARK_CKPT_MODELS) and os.listdir(SPARK_CKPT_MODELS):
        return "trained"
    if os.path.isdir(SPARK_CKPT_SPLITS) and os.listdir(SPARK_CKPT_SPLITS):
        return "etl"
    return None


_LEGACY_CKPT_DIRS = {
    "GATE": "/home/sagemaker-user/PS1G_OOS/checkpoints",
    "TVM": "/home/sagemaker-user/PS1T_OOS/checkpoints",
    "VALIDATOR": "/home/sagemaker-user/PS1V_OOS/checkpoints",
}


def _migrate_legacy_ckpt_if_needed():
    """One-time copy from old SageMaker home-dir layout into OUT_DIR/checkpoints."""
    _legacy = _LEGACY_CKPT_DIRS.get(globals().get("DEVICE_CAT", ""), "")
    if not _legacy or not os.path.isdir(_legacy) or _infer_ckpt_stage_from_disk():
        return
    os.makedirs(CKPT_DIR, exist_ok=True)
    for _name in os.listdir(_legacy):
        _src = os.path.join(_legacy, _name)
        _dst = os.path.join(CKPT_DIR, _name)
        if os.path.isdir(_src):
            shutil.copytree(_src, _dst, dirs_exist_ok=True)
        elif os.path.isfile(_src):
            shutil.copy2(_src, _dst)
    print(f"[CHECKPOINT] Migrated legacy checkpoints {_legacy} -> {CKPT_DIR}")


def _load_ckpt_meta(path=SPARK_CKPT_META):
    """Load checkpoint meta; return None if missing or corrupt."""
    if not path or not os.path.exists(path):
        return None
    try:
        if os.path.getsize(path) == 0:
            raise EOFError("empty checkpoint file")
        return joblib.load(path)
    except Exception as exc:
        _bak = path + ".corrupt"
        try:
            if os.path.exists(_bak):
                os.remove(_bak)
            os.rename(path, _bak)
        except Exception:
            pass
        print(f"[CHECKPOINT] Corrupt meta ({type(exc).__name__}) — moved to {_bak}")
        return None


def _save_ckpt_meta(meta, path=SPARK_CKPT_META):
    """Atomic joblib write — avoids truncated meta on crash."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    _tmp = path + ".tmp"
    joblib.dump(meta, _tmp)
    os.replace(_tmp, path)


def _rebuild_ckpt_meta_from_artifacts():
    """Rebuild minimal meta when joblib file is corrupt but parquet/models remain."""
    if not os.path.isdir(SPARK_CKPT_SPLITS):
        return None
    try:
        import pyarrow.parquet as pq
        _names = pq.read_schema(SPARK_CKPT_SPLITS).names
    except Exception as exc:
        print(f"[CHECKPOINT] Cannot read splits schema: {exc}")
        return None
    _key_cols = list(globals().get("KEY_COLS", ["DEVICE_ID", "transit_day"]))
    _id_cols = list(globals().get("ID_COLS", ["DEVICE_ID", "event_date"]))
    _skip = set(
        _key_cols + _id_cols + [
            TARGET_COL, TARGET, "split", "weight", "mars_device_category", "transit_day",
        ]
    )
    if "JOIN_KEY_DEVICE" in globals():
        _skip.add(JOIN_KEY_DEVICE)
    _features = [c for c in _names if c not in _skip]
    if not _features:
        return None
    _stage = _infer_ckpt_stage_from_disk() or "etl"
    _model_names = []
    if os.path.isdir(SPARK_CKPT_MODELS):
        _model_names = [
            n for n in os.listdir(SPARK_CKPT_MODELS)
            if os.path.isdir(os.path.join(SPARK_CKPT_MODELS, n))
        ]
    _champion = None
    for _pref in (
        "spark_xgb_optuna", "spark_xgb", "spark_lgb_optuna", "spark_lgb",
        "spark_catboost_optuna", "spark_catboost", "spark_gbt", "spark_rf", "spark_lr",
    ):
        if _pref in _model_names:
            _champion = _pref
            break
    if _champion is None and _model_names:
        _champion = sorted(_model_names)[0]
    _tag = DEVICE_CAT.lower()
    _cat_stub = {
        "scale_pos_weight": 1.0,
        "recall_floor": QUALITY_RECALL_FLOOR.get(DEVICE_CAT, 0.70),
        **ps1_resource_names(DEVICE_CAT),
    }
    _meta = {
        "version": SPARK_CKPT_VERSION,
        "stage": _stage,
        "FEATURE_COLS": _features,
        "ID_COLS": list(ID_COLS),
        "TARGET": TARGET,
        "TARGET_COL": TARGET_COL,
        "DEVICE_CAT": DEVICE_CAT,
        "DATA_SOURCE": "pyspark_sparkml",
        "CATEGORIES": {DEVICE_CAT: _cat_stub},
        "GOLD_VERSION": GOLD_VERSION,
        "rebuilt_from_artifacts": True,
    }
    if _stage in ("trained", "evaluated") and _model_names:
        _meta["models"] = {n: {} for n in _model_names}
        _meta["CHAMPION_NAME"] = _champion
    print(f"[CHECKPOINT] Rebuilt meta from artifacts: stage={_stage} features={len(_features)} champion={_champion}")
    _save_ckpt_meta(_meta)
    return _meta


def should_run_etl():
    return not (CHECKPOINT_LOADED and CHECKPOINT_STAGE in ("etl", "trained", "evaluated"))


def should_run_training():
    return not (CHECKPOINT_LOADED and CHECKPOINT_STAGE in ("trained", "evaluated"))


def ps1_resource_names(category=None):
    """Canonical SageMaker / MLflow resource names (single source of truth)."""
    category = category or DEVICE_CAT
    tag = category.lower()
    suffix = globals().get("PS1_NAME_SUFFIX", "failure")
    return {
        "experiment_name": f"chicago-ps1-3d-{tag}-{suffix}",
        "model_name": f"chicago-ps1-3d-{tag}-{suffix}",
        "registry_name": f"chicago-ps1-3d-{tag}-{suffix}",
        "fs_group": f"chicago-ps1-3d-{tag}-{suffix}-features",
        "endpoint": f"chicago-ps1-3d-{tag}-{suffix}-v1",
    }


def apply_ps1_resource_names(categories=None):
    """Refresh names on CATEGORIES — fixes legacy checkpoints (hardware-oos-sparkml)."""
    categories = categories if categories is not None else globals().get("CATEGORIES") or {}
    for cat, meta in categories.items():
        names = ps1_resource_names(cat)
        _prev_ep = meta.get("endpoint")
        _prev_model = meta.get("model_name")
        meta.update(names)
        if _prev_ep and _prev_ep != names["endpoint"]:
            print(f"[NAMES] {cat} endpoint: {_prev_ep} -> {names['endpoint']}")
        if _prev_model and _prev_model != names["model_name"]:
            print(f"[NAMES] {cat} model: {_prev_model} -> {names['model_name']}")
    return categories


def _save_spark_model_to_disk(name, model):
    """Write one PipelineModel immediately so JVM restarts can reload it."""
    os.makedirs(SPARK_CKPT_MODELS, exist_ok=True)
    mpath = os.path.join(SPARK_CKPT_MODELS, name)
    if os.path.exists(mpath):
        shutil.rmtree(mpath)
    ensure_spark_alive()
    model.write().overwrite().save(mpath)


def _rebind_spark_state_after_restart():
    """Reload splits/models from disk after SparkContext was recreated."""
    global CHAMPION_MODEL, CHAMPION_NAME, df_ml_spark, train_sdf, val_sdf, test_sdf, CATEGORIES
    if CHECKPOINT_LOADED and os.path.exists(SPARK_CKPT_META):
        restore_spark_checkpoint()
        return
    from pyspark.ml import PipelineModel
    if os.path.exists(SPARK_CKPT_SPLITS):
        df_ml_spark = spark.read.parquet(SPARK_CKPT_SPLITS)
        train_sdf = df_ml_spark.filter(F.col("split") == "train")
        val_sdf = df_ml_spark.filter(F.col("split") == "val")
        test_sdf = df_ml_spark.filter(F.col("split") == "test")
        if CATEGORIES:
            for _cat in CATEGORIES:
                CATEGORIES[_cat]["train_sdf"] = train_sdf
                CATEGORIES[_cat]["val_sdf"] = val_sdf
                CATEGORIES[_cat]["test_sdf"] = test_sdf
    if not os.path.isdir(SPARK_CKPT_MODELS):
        for _res in SPARK_ML_RESULTS.values():
            _res.pop("model", None)
            _res.pop("val_pred", None)
            _res.pop("test_pred", None)
        return
    _reloaded = {}
    for _name in os.listdir(SPARK_CKPT_MODELS):
        _mpath = os.path.join(SPARK_CKPT_MODELS, _name)
        if not os.path.isdir(_mpath):
            continue
        try:
            _model = PipelineModel.load(_mpath)
            _meta = {
                k: v for k, v in SPARK_ML_RESULTS.get(_name, {}).items()
                if k not in ("model", "val_pred", "test_pred")
            }
            _reloaded[_name] = {**_meta, "model": _model}
        except Exception as _exc:
            print(f"[WARN] Could not reload model {_name}: {_exc}")
    SPARK_ML_RESULTS.clear()
    SPARK_ML_RESULTS.update(_reloaded)
    if CHAMPION_NAME in SPARK_ML_RESULTS:
        CHAMPION_MODEL = SPARK_ML_RESULTS[CHAMPION_NAME]["model"]
    elif _reloaded:
        CHAMPION_NAME = max(_reloaded, key=lambda k: _reloaded[k].get("val_ap", -1))
        CHAMPION_MODEL = _reloaded[CHAMPION_NAME]["model"]
    if _reloaded:
        print(f"[WARN] Rebound {len(_reloaded)} model(s) from {SPARK_CKPT_MODELS}")


def save_spark_etl_checkpoint():
    """Persist Spark splits + metadata (call at end of CELL 9)."""
    global CHECKPOINT_STAGE
    ensure_spark_alive()
    apply_ps1_resource_names(CATEGORIES)
    os.makedirs(CKPT_DIR, exist_ok=True)
    df_ml_spark.write.mode("overwrite").parquet(SPARK_CKPT_SPLITS)
    meta = {
        "version": SPARK_CKPT_VERSION,
        "stage": "etl",
        "FEATURE_COLS": list(FEATURE_COLS),
        "ID_COLS": list(ID_COLS),
        "TARGET": TARGET,
        "TARGET_COL": TARGET_COL,
        "DEVICE_CAT": DEVICE_CAT,
        "DATA_SOURCE": DATA_SOURCE,
        "CATEGORIES": _serializable_categories(CATEGORIES),
        "GOLD_VERSION": GOLD_VERSION,
        "PS1_LABEL_SIGNATURE": dict(_PS1_LABEL_SIGNATURE),
    }
    _save_ckpt_meta(meta, SPARK_CKPT_META)
    CHECKPOINT_STAGE = "etl"
    print(f"[CHECKPOINT] ETL saved -> {SPARK_CKPT_SPLITS}  ({df_ml_spark.count():,} rows)")


def save_spark_training_checkpoint():
    """Persist fitted PipelineModels (call at end of CELL 12)."""
    global CHECKPOINT_STAGE
    ensure_spark_alive()
    apply_ps1_resource_names(CATEGORIES)
    os.makedirs(SPARK_CKPT_MODELS, exist_ok=True)
    models_meta = {}
    for name, res in SPARK_ML_RESULTS.items():
        mpath = os.path.join(SPARK_CKPT_MODELS, name)
        try:
            _model = res.get("model")
            if _model is not None and not os.path.exists(mpath):
                _model.write().overwrite().save(mpath)
            elif _model is None and not os.path.exists(mpath):
                print(f"[CHECKPOINT] Skip {name} — no model on disk")
                continue
        except Exception as _exc:
            _msg = str(_exc)
            if "stopped SparkContext" in _msg or "SparkContext" in _msg:
                print(f"[CHECKPOINT] Stale in-memory model {name} — using on-disk copy")
                if not os.path.exists(mpath):
                    continue
            else:
                raise
        models_meta[name] = {
            k: v for k, v in res.items()
            if k not in ("model", "val_pred", "test_pred")
        }
    meta = _load_ckpt_meta(SPARK_CKPT_META) or {}
    meta.update({
        "version": SPARK_CKPT_VERSION,
        "stage": "trained",
        "FEATURE_COLS": list(FEATURE_COLS),
        "ID_COLS": list(ID_COLS),
        "TARGET": TARGET,
        "TARGET_COL": TARGET_COL,
        "DEVICE_CAT": DEVICE_CAT,
        "DATA_SOURCE": DATA_SOURCE,
        "CATEGORIES": _serializable_categories(CATEGORIES),
        "GOLD_VERSION": GOLD_VERSION,
        "CHAMPION_NAME": CHAMPION_NAME,
        "models": models_meta,
        "CATEGORY_RESULTS": _serializable_category_results(CATEGORY_RESULTS),
    })
    _save_ckpt_meta(meta, SPARK_CKPT_META)
    CHECKPOINT_STAGE = "trained"
    print(f"[CHECKPOINT] Training saved -> {SPARK_CKPT_MODELS}  champion={CHAMPION_NAME}")


def save_spark_eval_checkpoint():
    """Persist threshold/QG + test_pdf (call at end of CELL 13)."""
    global CHECKPOINT_STAGE
    os.makedirs(SPARK_CKPT_EVAL, exist_ok=True)
    cat = DEVICE_CAT
    _res = CATEGORY_RESULTS.get(cat, {})
    _test_pdf = _res.get("test_pdf")
    if _test_pdf is not None and len(_test_pdf):
        _test_pdf.to_parquet(os.path.join(SPARK_CKPT_EVAL, "test_pdf.parquet"), index=False)
    meta = _load_ckpt_meta(SPARK_CKPT_META) or {}
    meta.update({
        "version": SPARK_CKPT_VERSION,
        "stage": "evaluated",
        "CHAMPION_NAME": CHAMPION_NAME,
        "opt_thresh": _res.get("opt_thresh"),
        "qg_pass": _res.get("qg_pass"),
        "qg_recall": _res.get("qg_recall"),
        "qg_floor": _res.get("qg_floor"),
        "test_ap": float(_res.get("test_ap", SPARK_ML_RESULTS.get(CHAMPION_NAME, {}).get("test_ap", 0))),
        "CATEGORY_RESULTS": _serializable_category_results(CATEGORY_RESULTS),
    })
    _save_ckpt_meta(meta, SPARK_CKPT_META)
    CHECKPOINT_STAGE = "evaluated"
    print(f"[CHECKPOINT] Eval saved -> {SPARK_CKPT_EVAL}")


def restore_spark_checkpoint():
    """Reload splits + models into Spark session (call at end of CELL 5)."""
    global df_ml_spark, train_sdf, val_sdf, test_sdf, CATEGORIES, SPARK_ML_RESULTS
    global CHAMPION_NAME, CHAMPION_MODEL, CATEGORY_RESULTS
    global FEATURE_COLS, ID_COLS, TARGET, DATA_SOURCE, CHECKPOINT_STAGE

    if not CHECKPOINT_LOADED:
        return False
    meta = _load_ckpt_meta(SPARK_CKPT_META)
    if meta is None:
        meta = _rebuild_ckpt_meta_from_artifacts()
    if meta is None:
        print("[CHECKPOINT] Meta missing — run full ETL.")
        return False

    if meta.get("version") != SPARK_CKPT_VERSION:
        raise RuntimeError(
            f"[CHECKPOINT] Unsupported version {meta.get('version')} — set FORCE_RELOAD=True"
        )

    FEATURE_COLS = list(meta["FEATURE_COLS"])
    ID_COLS = list(meta["ID_COLS"])
    TARGET = meta["TARGET"]
    DATA_SOURCE = meta.get("DATA_SOURCE", "pyspark_sparkml")
    CHECKPOINT_STAGE = meta["stage"]

    df_ml_spark = spark.read.parquet(SPARK_CKPT_SPLITS)
    train_sdf = df_ml_spark.filter(F.col("split") == "train")
    val_sdf = df_ml_spark.filter(F.col("split") == "val")
    test_sdf = df_ml_spark.filter(F.col("split") == "test")

    CATEGORIES = {}
    for cat, cat_meta in meta["CATEGORIES"].items():
        CATEGORIES[cat] = {
            **cat_meta,
            "train_sdf": train_sdf,
            "val_sdf": val_sdf,
            "test_sdf": test_sdf,
        }
    if DEVICE_CAT not in CATEGORIES and "TVM" in CATEGORIES:
        CATEGORIES[DEVICE_CAT] = CATEGORIES.pop("TVM")
        print(f"[CHECKPOINT] Migrated legacy CATEGORIES['TVM'] -> CATEGORIES[{DEVICE_CAT!r}]")
    apply_ps1_resource_names(CATEGORIES)
    ensure_mlflow_tracking()
    for cat in CATEGORIES:
        ensure_ps1_experiment(CATEGORIES[cat]["experiment_name"])

    SPARK_ML_RESULTS = {}
    CATEGORY_RESULTS = meta.get("CATEGORY_RESULTS", {})
    if DEVICE_CAT not in CATEGORY_RESULTS and "TVM" in CATEGORY_RESULTS:
        CATEGORY_RESULTS[DEVICE_CAT] = CATEGORY_RESULTS.pop("TVM")

    if CHECKPOINT_STAGE in ("trained", "evaluated") and meta.get("models"):
        from pyspark.ml import PipelineModel
        for name, minfo in meta["models"].items():
            mpath = os.path.join(SPARK_CKPT_MODELS, name)
            SPARK_ML_RESULTS[name] = {**minfo, "model": PipelineModel.load(mpath)}
        CHAMPION_NAME = meta["CHAMPION_NAME"]
        CHAMPION_MODEL = SPARK_ML_RESULTS[CHAMPION_NAME]["model"]
        if DEVICE_CAT in CATEGORY_RESULTS:
            CATEGORY_RESULTS[DEVICE_CAT]["champion"] = CHAMPION_NAME
            CATEGORY_RESULTS[DEVICE_CAT]["champion_model"] = CHAMPION_MODEL

    if CHECKPOINT_STAGE == "evaluated":
        _tp = os.path.join(SPARK_CKPT_EVAL, "test_pdf.parquet")
        if os.path.exists(_tp):
            _pdf = pd.read_parquet(_tp)
            if DEVICE_CAT not in CATEGORY_RESULTS:
                CATEGORY_RESULTS[DEVICE_CAT] = {}
            CATEGORY_RESULTS[DEVICE_CAT]["test_pdf"] = _pdf
            for _k in ("opt_thresh", "qg_pass", "qg_recall", "qg_floor", "test_ap"):
                if _k in meta:
                    CATEGORY_RESULTS[DEVICE_CAT][_k] = meta[_k]

    print(f"[CHECKPOINT] Restored stage={CHECKPOINT_STAGE}  features={len(FEATURE_COLS)}  "
          f"rows={df_ml_spark.count():,}  champion={meta.get('CHAMPION_NAME', '—')}")
    return True


def rehydrate_champion_predictions(category=None):
    """Recompute val/test preds if missing (after checkpoint restore)."""
    category = category or DEVICE_CAT
    if CHAMPION_NAME not in SPARK_ML_RESULTS:
        return
    _entry = SPARK_ML_RESULTS[CHAMPION_NAME]
    if "val_pred" in _entry and "test_pred" in _entry:
        return
    _model = _entry["model"]
    _cat = CATEGORIES[category]
    print(f"[CHECKPOINT] Rehydrating predictions for {CHAMPION_NAME} ...")
    _entry["val_pred"] = _model.transform(_cat["val_sdf"])
    _entry["test_pred"] = _model.transform(_cat["test_sdf"])


# ── Spark session health (available after CELL 4; CELL 11/12 call ensure_spark_alive) ─
import math
import subprocess
import sys
from pyspark.sql import SparkSession

spark = globals().get("spark")
SPARK_RUNTIME = globals().get("SPARK_RUNTIME", "unknown")
SPARK_MASTER = globals().get("SPARK_MASTER", "")
IS_LOCAL_SPARK = False
shuffle_partitions = LOCAL_SHUFFLE_PARTITIONS
JVM_MAX_HEAP_GB = float("nan")


def _run_conda(requirements):
    """Install into the active Conda/Jupyter environment."""
    args = " ".join(requirements)
    try:
        ip = get_ipython()  # noqa: F821
    except NameError:
        ip = None
    if ip is not None:
        ip.run_line_magic("conda", f"install -y -c conda-forge {args}")
    else:
        conda_executable = shutil.which("conda")
        if conda_executable is None:
            raise RuntimeError(
                "Conda is unavailable — run %conda install -y -c conda-forge openjdk=17"
            )
        subprocess.check_call([
            conda_executable, "install", "-y", "-c", "conda-forge", *requirements
        ])


def _java_from_home(java_home):
    if not java_home:
        return None
    candidate = os.path.join(java_home, "bin", "java")
    return candidate if os.path.isfile(candidate) and os.access(candidate, os.X_OK) else None


def _discover_java():
    """Return (JAVA_HOME, java executable), resolving system/Conda symlinks."""
    configured_home = os.environ.get("JAVA_HOME")
    configured_java = _java_from_home(configured_home)
    if configured_java:
        return os.path.realpath(configured_home), os.path.realpath(configured_java)
    candidates = [
        shutil.which("java"),
        os.path.join(sys.prefix, "bin", "java"),
        "/usr/bin/java",
        "/usr/lib/jvm/java-17-amazon-corretto/bin/java",
        "/usr/lib/jvm/java-11-amazon-corretto/bin/java",
    ]
    for candidate in candidates:
        if candidate and os.path.isfile(candidate) and os.access(candidate, os.X_OK):
            resolved_java = os.path.realpath(candidate)
            resolved_home = os.path.dirname(os.path.dirname(resolved_java))
            if _java_from_home(resolved_home):
                return resolved_home, resolved_java
            if candidate.startswith(os.path.realpath(sys.prefix) + os.sep):
                return os.path.realpath(sys.prefix), resolved_java
    return None, None


def ensure_local_java():
    """Prepare JDK for local Spark; auto-install OpenJDK on SageMaker if missing."""
    java_home, java_executable = _discover_java()
    if java_executable is None and AUTO_INSTALL_LOCAL_JAVA:
        print(f"Java absent — installing OpenJDK {OPENJDK_VERSION} into this kernel ...")
        _run_conda([f"openjdk={OPENJDK_VERSION}"])
        java_home, java_executable = _discover_java()
    if java_executable is None:
        raise RuntimeError(
            f"Local Spark requires Java. Run: %conda install -y -c conda-forge openjdk={OPENJDK_VERSION} "
            "then restart kernel; or use a SparkAnalytics / EMR kernel."
        )
    os.environ["JAVA_HOME"] = java_home
    java_bin = os.path.join(java_home, "bin")
    if java_bin not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = java_bin + os.pathsep + os.environ.get("PATH", "")
    _ver = subprocess.run(
        [java_executable, "-version"], text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=30, check=False,
    )
    if _ver.returncode != 0:
        raise RuntimeError(f"java -version failed at {java_executable}")
    _line = _ver.stdout.splitlines()[0] if _ver.stdout else "OK"
    print(f"JAVA_HOME: {java_home}  ({_line})")
    return java_home


def _apply_spark_runtime_conf(_spark):
    global SPARK_MASTER, IS_LOCAL_SPARK, shuffle_partitions, JVM_MAX_HEAP_GB
    SPARK_MASTER = _spark.sparkContext.master
    IS_LOCAL_SPARK = SPARK_MASTER.startswith("local")
    shuffle_partitions = (
        LOCAL_SHUFFLE_PARTITIONS
        if IS_LOCAL_SPARK
        else max(32, min(2000, _spark.sparkContext.defaultParallelism * 2))
    )
    _runtime_conf = {
        "spark.sql.adaptive.enabled": "true",
        "spark.sql.adaptive.coalescePartitions.enabled": "false" if IS_LOCAL_SPARK else "true",
        "spark.sql.adaptive.skewJoin.enabled": "true",
        "spark.sql.parquet.filterPushdown": "true",
        "spark.sql.files.maxPartitionBytes": str(64 * 1024 * 1024),
        "spark.sql.adaptive.advisoryPartitionSizeInBytes": str(32 * 1024 * 1024),
        "spark.sql.shuffle.partitions": str(shuffle_partitions),
        "spark.sql.execution.arrow.pyspark.enabled": "true",
        "spark.sql.execution.arrow.pyspark.fallback.enabled": "true",
        "spark.sql.execution.arrow.maxRecordsPerBatch": "10000",
        "spark.sql.debug.maxToStringFields": "200",
        "spark.sql.session.timeZone": "UTC",
        **({"spark.task.cpus": "1"} if IS_LOCAL_SPARK else {}),
    }
    for key, value in _runtime_conf.items():
        try:
            _spark.conf.set(key, value)
        except Exception as exc:
            print(f"Config retained by existing session: {key} ({type(exc).__name__})")
    _apply_s3a_hadoop_conf(_spark)
    _spark.sparkContext.setLogLevel("WARN")
    try:
        JVM_MAX_HEAP_GB = _spark._jvm.java.lang.Runtime.getRuntime().maxMemory() / (1024 ** 3)
    except Exception:
        JVM_MAX_HEAP_GB = float("nan")


def _apply_s3a_hadoop_conf(_spark):
    """Override S3A duration defaults like '60s' that break Hadoop 3.3.x parsers."""
    _s3a_numeric = {
        "fs.s3a.connection.timeout": "200000",
        "fs.s3a.connection.establish.timeout": "30000",
        "fs.s3a.connection.request.timeout": "60000",
        "fs.s3a.connection.acquisition.timeout": "60000",
        "fs.s3a.connection.idle.time": "60000",
        "fs.s3a.connection.ttl": "300000",
        "fs.s3a.socket.timeout": "60000",
        "fs.s3a.threads.keepalivetime": "60",
        "fs.s3a.multipart.purge.age": "86400000",
    }
    for key, value in _s3a_numeric.items():
        try:
            _spark.conf.set(f"spark.hadoop.{key}", value)
        except Exception as exc:
            print(f"[S3A] spark.conf {key}: {type(exc).__name__}")
    try:
        _hconf = _spark.sparkContext._jsc.hadoopConfiguration()
        for key, value in _s3a_numeric.items():
            _hconf.set(key, value)
    except Exception as exc:
        print(f"[S3A] hadoopConfiguration override: {type(exc).__name__}")
    try:
        _spark._jvm.org.apache.hadoop.fs.FileSystem.closeAll()
    except Exception:
        pass


def _create_local_spark_session(app_suffix=""):
    ensure_local_java()
    os.makedirs(LOCAL_SPARK_TEMP_DIR, exist_ok=True)
    existing_submit_args = os.environ.get("PYSPARK_SUBMIT_ARGS", "pyspark-shell")
    if "--driver-memory" not in existing_submit_args:
        if not existing_submit_args.strip().endswith("pyspark-shell"):
            existing_submit_args = existing_submit_args.strip() + " pyspark-shell"
        os.environ["PYSPARK_SUBMIT_ARGS"] = (
            f"--driver-memory {LOCAL_SPARK_DRIVER_MEMORY} {existing_submit_args}"
        )
    _app = f"PS1_{DEVICE_CAT}_OOS_MLflow_FeatureStore{app_suffix}"
    _builder = (
        SparkSession.builder
        .appName(_app)
        .master(f"local[{LOCAL_SPARK_THREADS}]")
        .config("spark.driver.memory", LOCAL_SPARK_DRIVER_MEMORY)
        .config("spark.driver.maxResultSize", f"{max(1, LOCAL_SPARK_DRIVER_MEMORY_GB // 2)}g")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("spark.local.dir", LOCAL_SPARK_TEMP_DIR)
        .config("spark.jars.packages", SPARK_JARS_PACKAGES)
        .config("spark.jars.repositories", SYNAPSEML_MAVEN_REPO)
        .config("spark.jars.excludes", SPARK_JARS_EXCLUDES)
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.DefaultAWSCredentialsProviderChain",
        )
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.connection.timeout", "200000")
        .config("spark.hadoop.fs.s3a.connection.establish.timeout", "30000")
        .config("spark.hadoop.fs.s3a.connection.request.timeout", "60000")
        .config("spark.hadoop.fs.s3a.connection.acquisition.timeout", "60000")
        .config("spark.hadoop.fs.s3a.connection.idle.time", "60000")
        .config("spark.hadoop.fs.s3a.connection.ttl", "300000")
        .config("spark.hadoop.fs.s3a.socket.timeout", "60000")
        .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60")
    )
    _sess = _builder.getOrCreate()
    _apply_s3a_hadoop_conf(_sess)
    return _sess


def ensure_mlflow_tracking():
    """Re-apply SageMaker MLflow ARN (required even if CELL 3 was skipped)."""
    os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN
    mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)


def ensure_ps1_experiment(experiment_name):
    """Create, restore (if soft-deleted), or activate a PS1 MLflow experiment."""
    from mlflow.tracking import MlflowClient

    ensure_mlflow_tracking()
    client = MlflowClient()
    exp = client.get_experiment_by_name(experiment_name)
    if exp is None:
        client.create_experiment(experiment_name)
        print(f"[MLflow] Created experiment: {experiment_name}")
    elif exp.lifecycle_stage == "deleted":
        client.restore_experiment(exp.experiment_id)
        print(f"[MLflow] Restored soft-deleted experiment: {experiment_name}")
    mlflow.set_experiment(experiment_name)


SPARK_JVM_BOOSTER_CLASSES = {
    "synapse_lgb": "com.microsoft.azure.synapse.ml.lightgbm.LightGBMClassifier",
    "catboost": "ai.catboost.spark.CatBoostClassifier",
}
SPARK_JAR_STATUS = {"synapse_lgb": False, "catboost": False}


def jvm_has_class(fqn, _spark=None):
    _spark = _spark or globals().get("spark")
    if _spark is None:
        return False
    try:
        _spark.sparkContext._jvm.java.lang.Class.forName(fqn)
        return True
    except Exception:
        return False


def probe_spark_booster_jars(_spark=None):
    """Return which distributed booster JARs are actually loaded in the JVM."""
    status = {k: jvm_has_class(v, _spark) for k, v in SPARK_JVM_BOOSTER_CLASSES.items()}
    globals()["SPARK_JAR_STATUS"] = status
    return status


def ensure_spark_alive(recreate=True):
    """Verify JVM/Py4J gateway; bootstrap or recreate session if needed."""
    global spark, SPARK_RUNTIME
    _spark = globals().get("spark")
    if _spark is not None:
        try:
            _spark.range(1).count()
            return _spark
        except Exception as _err:
            print(f"[WARN] Spark JVM not responding ({type(_err).__name__})")
    else:
        _active = SparkSession.getActiveSession()
        if _active is not None:
            spark = _active
            SPARK_RUNTIME = "managed/existing"
            _apply_spark_runtime_conf(spark)
            if CHECKPOINT_LOADED:
                restore_spark_checkpoint()
            return spark
        print("[WARN] No Spark session — starting local Spark (CELL 5 skipped) ...")
        spark = _create_local_spark_session()
        SPARK_RUNTIME = "local"
        _apply_spark_runtime_conf(spark)
        _rebind_spark_state_after_restart()
        return spark
    if not recreate:
        raise RuntimeError(
            "Spark JVM died — restart kernel, run Cells 1-4, then CELL 5 or training cells."
        )
    print("[WARN] Recreating Spark session (reload checkpoint if CHECKPOINT_LOADED=True) ...")
    try:
        if _spark is not None:
            _spark.stop()
    except Exception:
        pass
    try:
        SparkSession._instantiatedSession = None
    except Exception:
        pass
    spark = _create_local_spark_session(app_suffix="_restarted")
    SPARK_RUNTIME = "local"
    _apply_spark_runtime_conf(spark)
    _rebind_spark_state_after_restart()
    print("[WARN] Spark session recreated.")
    return spark


def spark_booster_profile():
    """Cap distributed boosters on local SageMaker to avoid JVM OOM."""
    if IS_LOCAL_SPARK:
        return (
            min(SPARK_LOCAL_MAX_WORKERS, LOCAL_SPARK_THREADS, spark.sparkContext.defaultParallelism),
            SPARK_LOCAL_BOOSTER_ROUNDS,
        )
    return max(1, spark.sparkContext.defaultParallelism), SPARK_CLUSTER_BOOSTER_ROUNDS

# ── Load checkpoint metadata (no Spark required yet) ─────────────────────────
_migrate_legacy_ckpt_if_needed()
# FORCE_RELOAD skips the RESTORE, but save_spark_training_checkpoint refuses to
# overwrite a model directory that already exists -- so without this a forced re-run
# reports its own metrics while silently keeping the PREVIOUS run's persisted models.
if FORCE_RELOAD and os.path.isdir(SPARK_CKPT_MODELS):
    # Move aside, never delete. On 24-Sep a FORCE_RELOAD that ran after a finished
    # training run removed its models outright, and the hour of compute went with them.
    # One discard is kept -- the previous one is removed -- so disk stays bounded.
    _discard = SPARK_CKPT_MODELS.rstrip("/") + ".discarded"
    if os.path.isdir(_discard):
        shutil.rmtree(_discard)
    shutil.move(SPARK_CKPT_MODELS, _discard)
    print(f"[CHECKPOINT] FORCE_RELOAD -- moved existing models aside to {_discard}")

if not FORCE_RELOAD and (os.path.exists(SPARK_CKPT_META) or _infer_ckpt_stage_from_disk()):
    _ckpt_meta = _load_ckpt_meta(SPARK_CKPT_META)
    if _ckpt_meta is None:
        _ckpt_meta = _rebuild_ckpt_meta_from_artifacts()
    if _ckpt_meta and _ckpt_meta.get("version") == SPARK_CKPT_VERSION:
        CHECKPOINT_STAGE = _ckpt_meta.get("stage")
        CHECKPOINT_LOADED = True
        FEATURE_COLS = list(_ckpt_meta["FEATURE_COLS"])
        ID_COLS = list(_ckpt_meta["ID_COLS"])
        TARGET = _ckpt_meta["TARGET"]
        print(f"[CHECKPOINT] Found stage={CHECKPOINT_STAGE} at {CKPT_DIR}")
        print("  Run CELL 5 (recommended) or jump to CELL 11/12 — ensure_spark_alive() restores Spark + checkpoint.")
        if CHECKPOINT_STAGE == "etl":
            print("  Then skip to CELL 10 (EDA) or CELL 11 (training).")
        elif CHECKPOINT_STAGE == "trained":
            print("  Then skip to CELL 13 (threshold/QG) or CELL 14+.")
        else:
            print("  Then skip to CELL 14+ (Feature Store / registration / deploy).")
            print("  For cross-wired export (CELL 24), re-run CELL 20 after restore.")
    elif _ckpt_meta is not None:
        print(f"[CHECKPOINT] Stale version {_ckpt_meta.get('version')} — set FORCE_RELOAD=True")
    else:
        print("[CHECKPOINT] No readable checkpoint meta — run Cells 5-12 or set FORCE_RELOAD=True")
else:
    print("[CHECKPOINT] No checkpoint or FORCE_RELOAD=True — run Cells 5-12 for full pipeline.")
    print(f"  Checkpoints will be written under {CKPT_DIR}")


# A restored checkpoint skips CELLS 6-13, so a changed configuration would be
# silently replaced by the PREVIOUS run's label, features and metrics. This has
# happened twice; the give-away both times was byte-identical output. Stop here
# rather than produce a plausible fake.
# A checkpoint built under a DIFFERENT configuration is just as dangerous as a
# different configuration meeting an old checkpoint -- and far more so, because the
# run that replays it looks clean and every publish fence stays open. Compare the
# stored signature with this run's and refuse on any difference.
_PS1_SIGNATURE_VERIFIED = False
if CHECKPOINT_LOADED and not FORCE_RELOAD:
    _stored_meta = _load_ckpt_meta(SPARK_CKPT_META) or {}
    _stored_sig = _stored_meta.get("PS1_LABEL_SIGNATURE")
    if _stored_sig is None:
        print("[CHECKPOINT] !! This checkpoint predates label stamping, so what it was "
              "built from is UNKNOWN. If it may have come from an experimental run, "
              "delete the checkpoint directory or set PS1_FORCE_RELOAD=true.")
    else:
        # Compare VALUES over the union of keys -- never the dicts themselves.
        #
        # Adding a name to _PS1_SIGNATURE_VARS grows the key set without changing any
        # configuration. An older checkpoint simply has no entry for the new flag, and
        # "no entry" means exactly what "" means: the flag was off. Comparing the dicts
        # directly made EVERY pre-existing checkpoint raise -- and raise with an EMPTY
        # list of differing variables, because _diff defaults both sides to "" and the
        # new keys then compare equal. A hard refusal naming nothing is unactionable and
        # reads as a bug rather than a guard. Branch on _diff and the two can never
        # disagree again.
        _new_keys = sorted(set(_PS1_LABEL_SIGNATURE) - set(_stored_sig))
        _diff = sorted(
            k for k in set(_stored_sig) | set(_PS1_LABEL_SIGNATURE)
            if _stored_sig.get(k, "") != _PS1_LABEL_SIGNATURE.get(k, "")
        )
        if _diff:
            raise RuntimeError(
                "[CHECKPOINT] REFUSING TO RUN. The checkpoint on disk was built under a "
                "DIFFERENT configuration from this run, and replaying it would skip "
                "CELLS 6-13 and publish that other run's label and metrics as if they were "
                "this one's:\n"
                + "\n".join(
                    f"    {_k}: checkpoint={_stored_sig.get(_k, '') or '<unset>'!r} "
                    f"now={_PS1_LABEL_SIGNATURE.get(_k, '') or '<unset>'!r}"
                    for _k in _diff
                )
                + "\n\nSet PS1_FORCE_RELOAD=true to rebuild, or delete the checkpoint "
                  "directory to discard the other configuration entirely."
            )
        # Built under EXACTLY this configuration, so replaying it reproduces this run
        # rather than substituting another. Record that: the non-default-config gate
        # below must not refuse a run already verified here, or a read-only inspection
        # becomes impossible without a full rebuild.
        _PS1_SIGNATURE_VERIFIED = True
        if _new_keys:
            print("[CHECKPOINT] Signature gained " + ", ".join(_new_keys)
                  + " since this checkpoint was written. All were off then and are off "
                    "now, so the feature matrix is unchanged.")
        print("[CHECKPOINT] Label signature matches this run -- replay verified.")

# Deliberately still refuses when the signature could NOT be verified: an unstamped
# checkpoint meeting a non-default configuration is the original hazard.
if (CHECKPOINT_LOADED and _PS1_CONFIG_NON_DEFAULT and not FORCE_RELOAD
        and not _PS1_SIGNATURE_VERIFIED):
    raise RuntimeError(
        "[CHECKPOINT] REFUSING TO RUN. A checkpoint at stage "
        f"'{CHECKPOINT_STAGE}' was restored, but this run sets a non-default "
        "configuration:\n"
        + "\n".join(
            f"    {_k}={os.environ[_k]}"
            for _k in ("PS1_EVENT_DEFINITION", "PS1_LABEL_HORIZON_DAYS",
                       "PS1_EXCLUDE_RELIEVED", "PS1_EVENT_SESSION_GAP_DAYS",
                       "PS1_LABEL_MODE", "PS1_COMPLETENESS_GUARD",
                       "PS1_COVERAGE_SIGNAL", "PS1_COVERAGE_FLOOR",
                       "PS1_COVERAGE_LOOKBACK_DAYS", "PS1_COVERAGE_CONTROL_MONTH",
                       "PS1_LABEL_OBSERVED_EDGE", "PS1_EXCLUDE_ACTIVE_EPISODE",
                       "PS1_LABEL_MASK_UNOBSERVED",
                       *_PS1_FEATURE_FLAGS)
            if _k in os.environ
        )
        + "\n\nCELLS 6-13 would be SKIPPED and the previous run's metrics reported "
        "as if they were new.\nSet PS1_FORCE_RELOAD=true in the first cell, restart "
        "the kernel, and run again."
    )


In [ ]:
#  CELL 5 : PySpark session + feature lists (OOS Optimized ETL)
import importlib.util
import math
import os
import subprocess
import sys
def _run_pip(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
if importlib.util.find_spec("pyspark") is None:
    print("Installing pyspark into this kernel...")
    _run_pip([f"pyspark=={PYSPARK_VERSION}"])

# Feature families — VALIDATOR port (PySpark + Spark ML; aligned with GATE/TVM)
# S24 incident features EXCLUDED (~0% VALIDATOR device-day coverage).

FAILURE_FEATURES = [
    "roll_fail_7d", "roll_fail_30d", "roll_fail_90d",
    "days_since_fail", "fail_free_streak",
]

PS1_TAP_FEATURES = [
    "tap_count", "tap_reject_rate_pct", "peak_hour_tap_count",
    "tap_timeout_rate_pct", "tap_timeout_count",
]

# Prior-only rolling tap features (CELL 7 — silver read_tap_device_daily S30)
VALIDATOR_TAP_FEATURES = [
    "tvm_read_count_prior_7d", "tvm_read_count_prior_30d", "tvm_read_count_prior_90d",
    "tvm_reject_rate_prior_7d", "tvm_reject_rate_prior_30d", "tvm_reject_rate_prior_90d",
    "tvm_reject_rate_trend_7v30d", "tvm_null_rate_prior_7d",
]

VALIDATOR_GOLD_EXTRA = [
    "printer_events", "bankcard_events", "bhu_events", "chu_events", "scrst_events",
    "system_events", "comms_events", "csc_reader_events",
    "hardware_oos_count", "chargeable_outage_count", "chargeable_outage_min",
    "events_7d", "events_30d", "oos_events_7d", "hardware_oos_events_7d",
    "outage_min_7d", "availability_pct_7d", "critical_events_7d",
    "metric_p95_txn_ms", "metric_slow_tap_count", "metric_rolling_7d_avg_ms", "metric_txn_delta",
    "use_revenue_cents", "sales_7d_avg", "kpi_count",
]

STATION_CASCADE_FEATURES = [
    "station_cascade_days_30d", "station_coordinated_fail_days_30d", "days_since_station_cascade",
]

METRIC_FEATURES = [
    "met_avail_pct", "met_downtime_min", "met_event_count", "met_p95_downtime",
]

CHAIN_FEATURES = [
    "chain_length", "chain_failure_count", "inter_failure_days_mean",
    "inter_failure_days_min", "failure_acceleration_rate", "chain_event_diversity",
    "max_chain_downtime_min", "chain_component_diversity",
]

ANOMALY_FEATURES = [
    "anomaly_days_7d", "anomaly_days_30d", "reject_spike_flag_7d",
    "reject_spike_flag_30d", "max_hourly_reject_rate_7d",
    "stddev_hourly_reject_7d", "anomaly_score_7d", "anomaly_score_30d",
]

LIFETIME_FEATURES = [
    "device_age_days", "cumulative_taps_lifetime", "cumulative_failures_lifetime",
    "time_since_last_maintenance_days", "mttr_mean_days", "mttr_p90_days",
]

USAGE_FEATURES = ["days_in_service", "total_taps_lifetime", "tap_rate_30d"]

MTTR_FEATURES = [
    "mttr_avg_downtime_30d", "mttr_avg_downtime_90d", "mttr_failure_days_30d",
    "mttr_failure_days_90d", "mttr_days_since_prev_failure", "mttr_max_downtime_30d",
]

USAGE_EXT_FEATURES = [
    "usage_days_in_service", "usage_days_since_last_failure",
    "usage_cumulative_tap_count", "usage_cumulative_failure_count",
    "usage_cumulative_outage_min", "usage_daily_maint_events",
]

SPARK_DERIVED_FEATURES = [
    "printer_events_prior_sum_7d", "printer_events_prior_sum_30d",
    "bankcard_events_prior_sum_7d", "bankcard_events_prior_sum_30d",
    "bhu_events_prior_sum_7d", "bhu_events_prior_sum_30d",
    "chu_events_prior_sum_7d", "chu_events_prior_sum_30d",
    "scrst_events_prior_sum_7d", "scrst_events_prior_sum_30d",
    "comms_events_prior_sum_7d", "comms_events_prior_sum_30d",
    "chargeable_outage_count_prior_sum_7d", "chargeable_outage_count_prior_sum_30d",
    "hardware_oos_count_prior_sum_7d", "facility_peer_hw_oos_7d",
]

OOS_LABEL_CONCURRENT_FEATURES = [
    "oos_event_count", "hardware_oos_count", "oos_events_7d", "hardware_oos_events_7d",
    "event_count", "critical_events", "printer_events", "bankcard_events", "bhu_events",
    "chu_events", "scrst_events", "comms_events", "system_events", "csc_reader_events",
    "chargeable_outage_count", "chargeable_outage_min",
    "events_7d", "events_30d", "outage_min_7d", "critical_events_7d",
]

ALL_CANDIDATE_FEATURES = [
    c for c in list(dict.fromkeys(
        FAILURE_FEATURES + PS1_TAP_FEATURES + VALIDATOR_TAP_FEATURES + VALIDATOR_GOLD_EXTRA
        + METRIC_FEATURES + CHAIN_FEATURES + ANOMALY_FEATURES
        + LIFETIME_FEATURES + USAGE_FEATURES + MTTR_FEATURES + USAGE_EXT_FEATURES
        + SPARK_DERIVED_FEATURES + STATION_CASCADE_FEATURES
    )) if c not in OOS_LABEL_CONCURRENT_FEATURES
]

REQUIRED_PS1_FEATURES = FAILURE_FEATURES
KEY_COLS = ["DEVICE_ID", "transit_day"]
JOIN_KEY_DEVICE = "DEVICE_KEY"  # silver S20/S28 join key (kept on spine, not a model feature)

assert len(ALL_CANDIDATE_FEATURES) == len(set(ALL_CANDIDATE_FEATURES)), "Duplicate feature names"
print(f"VALIDATOR OOS candidate features: {len(ALL_CANDIDATE_FEATURES)} (excluded {len(OOS_LABEL_CONCURRENT_FEATURES)} concurrent)")



active_spark = SparkSession.getActiveSession()

if active_spark is not None:
    spark = active_spark
    SPARK_RUNTIME = "managed/existing"
else:
    spark = _create_local_spark_session()
    SPARK_RUNTIME = "local"

_apply_spark_runtime_conf(spark)
_jar_status = probe_spark_booster_jars(spark)
print(f"[Spark JARs] synapse_lgb={_jar_status['synapse_lgb']} catboost={_jar_status['catboost']}")
if IS_LOCAL_SPARK and not _jar_status["synapse_lgb"]:
    print("[WARN] SynapseML missing from JVM — recreating local Spark with booster JARs ...")
    try:
        spark.stop()
    except Exception:
        pass
    try:
        SparkSession._instantiatedSession = None
    except Exception:
        pass
    spark = _create_local_spark_session(app_suffix="_boosters")
    SPARK_RUNTIME = "local"
    _apply_spark_runtime_conf(spark)
    _jar_status = probe_spark_booster_jars(spark)
    print(f"[Spark JARs] after recreate: synapse_lgb={_jar_status['synapse_lgb']} catboost={_jar_status['catboost']}")
elif not IS_LOCAL_SPARK and not _jar_status["synapse_lgb"]:
    print("[WARN] Managed Spark lacks SynapseML JAR — spark_lgb will be skipped (XGB/RF/GBT still run).")
try:
    JVM_MAX_HEAP_GB = (
        spark._jvm.java.lang.Runtime.getRuntime().maxMemory() / (1024 ** 3)
    )
except Exception:
    JVM_MAX_HEAP_GB = float("nan")
if (
    IS_LOCAL_SPARK
    and math.isfinite(JVM_MAX_HEAP_GB)
    and JVM_MAX_HEAP_GB < max(0.8, LOCAL_SPARK_DRIVER_MEMORY_GB * 0.75)
):
    raise RuntimeError(
        f"The active local Spark JVM has only {JVM_MAX_HEAP_GB:.2f} GiB max heap, but "
        f"{LOCAL_SPARK_DRIVER_MEMORY} was requested. Restart the kernel and run from Cell 1 "
        "so the heap setting is applied before Spark starts."
    )


def spark_path(uri):
    if IS_LOCAL_SPARK and uri.startswith("s3://"):
        return "s3a://" + uri[len("s3://"):]
    return uri


S3_SILVER_RUNTIME = spark_path(S3_SILVER)
S3_GOLD_RUNTIME = spark_path(S3_GOLD)

print(f"Spark version       : {spark.version}")
print(f"Spark runtime       : {SPARK_RUNTIME}")
print(f"Spark master        : {SPARK_MASTER}")
print(f"Default parallelism : {spark.sparkContext.defaultParallelism}")
print(f"Shuffle partitions  : {shuffle_partitions}")
print(
    f"JVM max heap        : {JVM_MAX_HEAP_GB:.2f} GiB"
    if math.isfinite(JVM_MAX_HEAP_GB)
    else "JVM max heap        : unavailable from this managed session"
)
if IS_LOCAL_SPARK:
    print("WARNING: local Spark uses only this SageMaker instance; use EMR for the full-scale workload.")

# ── Restore Spark checkpoint (after session is live) ──────────────────────────
if CHECKPOINT_LOADED:
    restore_spark_checkpoint()


In [ ]:
#  CELL 7B : Load ps1_features into the notebook namespace
# =============================================================================
# CELL 8 previously relied on `# MAGIC %run ../ps1_features`. That line only
# does anything on Databricks; in SageMaker Studio it is an inert comment, so
# read_spine / add_auxiliary / join_and_materialise were never defined and
# CELL 8 died with:  NameError: name 'read_spine' is not defined
#
# Why exec() and not `import ps1_features`:
#   %run executes the file IN THE CALLER'S NAMESPACE. ps1_features relies on that
#   -- its helpers call _notebook_flag("IS_LOCAL_SPARK") and
#   _notebook_val("shuffle_partitions"), which read globals(). Under a plain
#   import those resolve against the MODULE's defaults, not against what
#   CELLS 3-5 just configured, so a local[*] session would silently be treated as
#   a cluster: no repartition by DEVICE_ID, and MEMORY_AND_DISK caching instead of
#   DISK_ONLY on a ~12 GiB local JVM.
#
# ps1_features also re-declares IS_LOCAL_SPARK=False and shuffle_partitions at
# module level, which would overwrite the values CELL 5 detected. The notebook's
# own values are therefore snapshotted and restored after the exec, and anything
# restored is printed so the override is visible rather than silent.
# =============================================================================
import os

_PS1_FEATURES_CANDIDATES = [
    os.environ.get("PS1_FEATURES_PATH", ""),
    os.path.join("..", "ps1_features.py"),
    "ps1_features.py",
    os.path.join("..", "..", "notebooks", "ps1_features.py"),
    os.path.join(os.path.expanduser("~"), "Chicago-Ventra-Mars-Cubic-Analysis",
                 "notebooks", "ps1_features.py"),
]
_ps1_features_path = next(
    (os.path.abspath(p) for p in _PS1_FEATURES_CANDIDATES if p and os.path.isfile(p)),
    None,
)
if _ps1_features_path is None:
    raise FileNotFoundError(
        "ps1_features.py not found. Looked in:\n  "
        + "\n  ".join(p for p in _PS1_FEATURES_CANDIDATES if p)
        + f"\nCurrent working directory is {os.getcwd()}.\n"
        "Fix: run this notebook from its own folder in the git clone, or set\n"
        "     os.environ['PS1_FEATURES_PATH'] = '/full/path/to/ps1_features.py'"
    )

# Names ps1_features assigns at module level. CELLS 3-5 are authoritative for any
# of these that already exist in this notebook.
_PS1_SHARED_CONFIG = [
    "TARGET_COL", "SLA_TARGET_COL", "LABEL_HORIZON_DAYS", "LABEL_DATA_LAG_DAYS",
    "KEY_COLS", "JOIN_KEY_DEVICE", "PS5_DATE_COLUMN", "AUXILIARY_DUPLICATE_POLICY",
    "RUN_DEEP_GRAIN_AUDIT", "IS_LOCAL_SPARK", "shuffle_partitions",
]
_ps1_preexisting = {k: globals()[k] for k in _PS1_SHARED_CONFIG if k in globals()}

with open(_ps1_features_path, "r", encoding="utf-8") as _fh:
    _ps1_src = _fh.read()
exec(compile(_ps1_src, _ps1_features_path, "exec"), globals())   # %run equivalent

_ps1_overridden = {}
for _k, _v in _ps1_preexisting.items():
    try:
        _changed = globals().get(_k) != _v
    except Exception:
        _changed = True
    if _changed:
        _ps1_overridden[_k] = _v
globals().update(_ps1_preexisting)

_PS1_REQUIRED = ["read_spine", "add_auxiliary", "join_and_materialise"]
_ps1_missing = [n for n in _PS1_REQUIRED if not callable(globals().get(n))]
if _ps1_missing:
    raise ImportError(
        f"ps1_features loaded from {_ps1_features_path} but these are still "
        f"undefined: {_ps1_missing}. The file on disk may be truncated."
    )

print(f"[ps1_features] loaded   -> {_ps1_features_path}")
print(f"[ps1_features] defined  -> {', '.join(_PS1_REQUIRED)}")
if _ps1_overridden:
    for _k, _v in _ps1_overridden.items():
        print(f"[ps1_features] kept notebook value  {_k} = {_v!r}")
else:
    print("[ps1_features] notebook config already matched the module defaults")


In [ ]:
#  CELL 6 : Gold/silver reads + hardware OOS label (PySpark)
# ps1_features helpers are loaded by CELL 7B above (the Databricks `%run ../ps1_features` does not execute in SageMaker Studio).
if should_run_etl():
    df_ps1 = read_spine(
        spark,
        DEVICE_CAT,
        TRAIN_START,
        None,
        with_label=True,
        s3_gold=S3_GOLD_RUNTIME,
        s3_silver=S3_SILVER_RUNTIME,
    )
else:
    print("[CHECKPOINT] Skip CELL 6 ETL — restored/saved checkpoint.")


In [ ]:
#  CELL 7 : Auxiliary reads + prior-window tap rolling (PySpark)
if should_run_etl():
    _ps1_aux = add_auxiliary(
        spark,
        df_ps1,
        DEVICE_CAT,
        TRAIN_START,
        None,
        s3_gold=S3_GOLD_RUNTIME,
        s3_silver=S3_SILVER_RUNTIME,
    )
    df_ps2 = _ps1_aux["df_ps2"]
    df_ps4 = _ps1_aux["df_ps4"]
    df_ps5 = _ps1_aux["df_ps5"]
    df_metric = _ps1_aux["df_metric"]
    df_mttr = _ps1_aux["df_mttr"]
    df_usage_ext = _ps1_aux["df_usage_ext"]
    df_read_tap = _ps1_aux["df_read_tap"]
else:
    print("[CHECKPOINT] Skip CELL 7 ETL — restored/saved checkpoint.")


In [ ]:
#  CELL 8 : Joins, grain audit, materialize features (PySpark)
if should_run_etl():
    df_features, FEATURE_COLS = join_and_materialise(
        spark,
        {
            "df_ps1": df_ps1,
            "df_ps2": df_ps2,
            "df_ps4": df_ps4,
            "df_ps5": df_ps5,
            "df_metric": df_metric,
            "df_mttr": df_mttr,
            "df_usage_ext": df_usage_ext,
            "df_read_tap": df_read_tap,
        },
        DEVICE_CAT,
        with_label=True,
        materialize=True,
    )
else:
    print("[CHECKPOINT] Skip CELL 8 ETL — restored/saved checkpoint.")


In [ ]:
#  CELL 9 : Spark temporal splits + feature pruning (stay in Spark — no toPandas)
if should_run_etl():
    TARGET = TARGET_COL
    _VAL_START = "2025-10-01"
    _TEST_START = "2026-01-01"
    _EMBARGO_DAYS = 14

    df_ml_spark = df_features.withColumn("event_date", F.col("transit_day"))
    if "mars_device_category" not in df_ml_spark.columns:
        df_ml_spark = df_ml_spark.withColumn("mars_device_category", F.lit(DEVICE_CAT))

    _embargo_val = F.date_sub(F.lit(_VAL_START), _EMBARGO_DAYS)
    _embargo_test = F.date_sub(F.lit(_TEST_START), _EMBARGO_DAYS)
    df_ml_spark = df_ml_spark.withColumn(
        "split",
        F.when(F.col("event_date") >= F.lit(_TEST_START), F.lit("test"))
        .when(
            (F.col("event_date") >= F.lit(_VAL_START)) & (F.col("event_date") < _embargo_test),
            F.lit("val"),
        )
        .when(F.col("event_date") < _embargo_val, F.lit("train"))
        .otherwise(F.lit("drop")),
    ).filter(F.col("split") != "drop")

    train_sdf = df_ml_spark.filter(F.col("split") == "train")
    val_sdf = df_ml_spark.filter(F.col("split") == "val")
    test_sdf = df_ml_spark.filter(F.col("split") == "test")

    with timed_stage("spark_prune_null_constant"):
        _null_exprs = [
            F.avg(F.when(F.col(c).isNull(), 1.0).otherwise(0.0)).alias(c) for c in FEATURE_COLS
        ]
        _null_row = train_sdf.agg(*_null_exprs).collect()[0].asDict()
        _dead = [c for c, r in _null_row.items() if r is not None and r > NULL_DROP_THRESH]
        ACTIVE_FEATURE_COLS = [c for c in FEATURE_COLS if c not in _dead]
        _const = []
        for c in ACTIVE_FEATURE_COLS:
            _stats = train_sdf.agg(
                F.approx_count_distinct(c, rsd=0.05).alias("n"),
                F.avg(F.col(c).cast("double")).alias("m"),
            ).collect()[0]
            if _stats["n"] is not None and int(_stats["n"]) <= 1:
                _const.append(c)
        ACTIVE_FEATURE_COLS = [c for c in ACTIVE_FEATURE_COLS if c not in _const]
        if not ACTIVE_FEATURE_COLS:
            raise RuntimeError("Every feature is null or constant on train split.")

    FEATURE_COLS = list(ACTIVE_FEATURE_COLS)
    ID_COLS = ["DEVICE_ID", "event_date"] + [
        c for c in ["DEVICE_KEY", "mars_device_category"] if c in df_ml_spark.columns
    ]

    with timed_stage("spark_split_counts"):
        _split_counts = {
            r["split"]: r["cnt"]
            for r in df_ml_spark.groupBy("split").count().withColumnRenamed("count", "cnt").collect()
        }
        _pos_neg = train_sdf.groupBy(TARGET_COL).count().collect()
        _pos = next((r["count"] for r in _pos_neg if r[TARGET_COL] == 1), 0)
        _neg = next((r["count"] for r in _pos_neg if r[TARGET_COL] == 0), 0)
        _spw = round(_neg / max(_pos, 1), 2)

    CATEGORIES = {
        DEVICE_CAT: {
            "train_sdf": train_sdf,
            "val_sdf": val_sdf,
            "test_sdf": test_sdf,
            "scale_pos_weight": _spw,
            "neg": _neg,
            "pos": _pos,
            "recall_floor": QUALITY_RECALL_FLOOR.get(DEVICE_CAT, 0.80),
            **ps1_resource_names(DEVICE_CAT),
        }
    }
    ensure_mlflow_tracking()
    ensure_ps1_experiment(CATEGORIES[DEVICE_CAT]["experiment_name"])
    SPARK_ML_RESULTS = {}
    DATA_SOURCE = "pyspark_sparkml"
    print(f"Spark splits — train={_split_counts.get('train',0):,} val={_split_counts.get('val',0):,} test={_split_counts.get('test',0):,}")
    print(f"Active features: {len(FEATURE_COLS)}  pos={_pos:,}  scale_pos_weight={_spw}")
    save_spark_etl_checkpoint()
else:
    print("[CHECKPOINT] Skip CELL 9 splits — restored/saved checkpoint.")


In [ ]:
#  CELL 10 : Spark EDA (aggregates; optional small sample for plots)
with timed_stage("spark_eda"):
    _rate = df_ml_spark.groupBy(TARGET_COL).count().collect()
    _daily = (
        df_ml_spark.groupBy("event_date")
        .agg(F.sum(TARGET_COL).alias("pos"), F.count("*").alias("n"))
        .orderBy("event_date")
    )
    print("Target counts:", {r[TARGET_COL]: r["count"] for r in _rate})
    print("Positive rate:", df_ml_spark.agg(F.avg(TARGET_COL)).collect()[0][0])
    try:
        _sample = _daily.limit(5000).toPandas()
        fig, ax = plt.subplots(figsize=(10, 3))
        _sample.set_index("event_date")["pos"].plot(ax=ax, color="#E53935", linewidth=0.8)
        ax.set_title(f"Daily OOS label count ({DEVICE_CAT})")
        plt.tight_layout()
        plt.show()
    except Exception as _e:
        print(f"[EDA] Plot skipped: {_e}")


In [ ]:
#  CELL 11 : Spark ML baselines — LR, RF, GBT, XGB, LGB, CatBoost (distributed)
if should_run_training():
    ensure_mlflow_tracking()
    from pyspark.ml.feature import Imputer, VectorAssembler
    from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
    from pyspark.ml.evaluation import BinaryClassificationEvaluator
    from pyspark.ml import Pipeline
    import mlflow.spark

    try:
        from xgboost.spark import SparkXGBClassifier
        _HAS_SPARK_XGB = True
    except ImportError:
        _HAS_SPARK_XGB = False
        print("[WARN] xgboost.spark not available — pip install 'xgboost==2.1.3'")

    try:
        from synapse.ml.lightgbm import LightGBMClassifier
        _HAS_SPARK_LGB = True
    except ImportError:
        _HAS_SPARK_LGB = False
        print("[WARN] SynapseML LightGBM not available — check spark.jars.packages (synapseml)")

    try:
        import catboost_spark
        from catboost_spark import CatBoostClassifier as SparkCatBoostClassifier
        _HAS_SPARK_CB = True
    except ImportError:
        _HAS_SPARK_CB = False
        print("[WARN] catboost_spark not available — check CatBoost Spark JAR in CELL 5")

    ensure_spark_alive()
    _jar_status = probe_spark_booster_jars(spark)
    if _HAS_SPARK_LGB and not _jar_status.get("synapse_lgb"):
        _HAS_SPARK_LGB = False
        print(
            "[WARN] SynapseML Python OK but JVM LightGBMClassifier missing — skipping spark_lgb. "
            "Restart kernel → CELL 5, or set SPARK_LOCAL_SKIP_SYNAPSE_LGB=True."
        )
    if _HAS_SPARK_CB and not _jar_status.get("catboost"):
        _HAS_SPARK_CB = False
        print(
            "[WARN] CatBoost Python OK but JVM CatBoostClassifier missing — skipping spark_catboost."
        )
    _NUM_WORKERS, _BOOSTER_ROUNDS = spark_booster_profile()
    if IS_LOCAL_SPARK:
        print(f"[local Spark] booster caps: workers={_NUM_WORKERS} rounds={_BOOSTER_ROUNDS}")

    def _use_spark_lgb():
        return _HAS_SPARK_LGB and not (IS_LOCAL_SPARK and SPARK_LOCAL_SKIP_SYNAPSE_LGB)

    def _use_spark_cb():
        return _HAS_SPARK_CB and not (IS_LOCAL_SPARK and SPARK_LOCAL_SKIP_SYNAPSE_CATBOOST)

    def _spark_lgb_num_tasks():
        return 1 if IS_LOCAL_SPARK else _NUM_WORKERS

    def _spark_lgb_kwargs(**overrides):
        kw = {
            "objective": "binary",
            "featuresCol": "features",
            "labelCol": TARGET_COL,
            "weightCol": "weight",
            **SPARK_LGB_KWARGS,
            "numTasks": _spark_lgb_num_tasks(),
        }
        kw.update(overrides)
        return kw

    if IS_LOCAL_SPARK and SPARK_LOCAL_SKIP_SYNAPSE_LGB:
        print("[local Spark] SynapseML LightGBM skipped (JVM SIGSEGV risk) — use EMR or SPARK_LOCAL_SKIP_SYNAPSE_LGB=False")
    if IS_LOCAL_SPARK and SPARK_LOCAL_SKIP_SYNAPSE_CATBOOST:
        print("[local Spark] CatBoost-Spark skipped — use EMR or SPARK_LOCAL_SKIP_SYNAPSE_CATBOOST=False")


    COMMON_TAGS = {"device_category": DEVICE_CAT, "target": TARGET_COL, "pipeline": "spark_ml_boosters", "run_date": pd.Timestamp.today().strftime("%Y-%m-%d")}

    def _add_weight(sdf, spw):
        return sdf.withColumn(
            "weight",
            F.when(F.col(TARGET_COL) == 1, F.lit(float(spw))).otherwise(F.lit(1.0)),
        )

    def _build_stages(classifier, feature_cols=None):
        cols = feature_cols or FEATURE_COLS
        imp_out = [f"{c}__imp" for c in cols]
        return [
            Imputer(inputCols=cols, outputCols=imp_out, strategy="median"),
            VectorAssembler(inputCols=imp_out, outputCol="features", handleInvalid="keep"),
            classifier,
        ]

    def _eval_ap(model, sdf):
        pred = model.transform(sdf)
        ev = BinaryClassificationEvaluator(
            labelCol=TARGET_COL, rawPredictionCol="rawPrediction", metricName="areaUnderPR"
        )
        auc_ev = BinaryClassificationEvaluator(
            labelCol=TARGET_COL, rawPredictionCol="rawPrediction", metricName="areaUnderROC"
        )
        return float(ev.evaluate(pred)), float(auc_ev.evaluate(pred)), pred

    def _train_and_log(name, classifier, train_sdf, val_sdf, test_sdf, spw):
        pipe = Pipeline(stages=_build_stages(classifier))
        with mlflow.start_run(run_name=name) as run:
            mlflow.set_tags({**COMMON_TAGS, "model_type": name})
            with timed_stage(f"fit_{name}"):
                model = pipe.fit(train_sdf)
            val_ap, val_auc, val_pred = _eval_ap(model, val_sdf)
            test_ap, test_auc, test_pred = _eval_ap(model, test_sdf)
            mlflow.log_params({
                "scale_pos_weight": spw, "n_features": len(FEATURE_COLS),
                "spark_master": spark.sparkContext.master, "num_workers": _NUM_WORKERS,
            })
            mlflow.log_metrics({
                "val_ap": val_ap, "val_auc": val_auc, "test_ap": test_ap, "test_auc": test_auc,
            })
            mlflow.spark.log_model(model, "model")
            SPARK_ML_RESULTS[name] = {
                "model": model, "run_id": run.info.run_id,
                "val_ap": val_ap, "test_ap": test_ap,
                "val_pred": val_pred, "test_pred": test_pred,
            }
            try:
                _save_spark_model_to_disk(name, model)
            except Exception as _ckpt_exc:
                print(f"[WARN] Incremental checkpoint for {name} failed: {_ckpt_exc}")
            print(f"{name}: val_ap={val_ap:.4f} test_ap={test_ap:.4f}")
            return val_ap

    CATEGORY = DEVICE_CAT
    cat = CATEGORIES[CATEGORY]
    _tr = _add_weight(cat["train_sdf"], cat["scale_pos_weight"])
    _va = cat["val_sdf"]
    _te = cat["test_sdf"]
    _spw = cat["scale_pos_weight"]

    _candidates = [
        ("spark_lr", LogisticRegression(
            featuresCol="features", labelCol=TARGET_COL, weightCol="weight",
            maxIter=50, regParam=0.01, elasticNetParam=0.15,
        )),
        ("spark_rf", RandomForestClassifier(
            featuresCol="features", labelCol=TARGET_COL, weightCol="weight",
            numTrees=200, maxDepth=12, minInstancesPerNode=50, seed=42,
        )),
        ("spark_gbt", GBTClassifier(
            featuresCol="features", labelCol=TARGET_COL, weightCol="weight",
            maxIter=80, maxDepth=6, stepSize=0.05, minInstancesPerNode=100, seed=42,
        )),
    ]

    if _HAS_SPARK_XGB:
        _candidates.append(("spark_xgb", SparkXGBClassifier(
            features_col="features", label_col=TARGET_COL, weight_col="weight",
            num_workers=_NUM_WORKERS, n_estimators=_BOOSTER_ROUNDS, max_depth=6, learning_rate=0.05,
            subsample=0.75, colsample_bytree=0.75, eval_metric="aucpr", seed=42,
        )))

    if _use_spark_lgb():
        _candidates.append(("spark_lgb", LightGBMClassifier(
            **_spark_lgb_kwargs(
                learningRate=0.05, numIterations=_BOOSTER_ROUNDS, numLeaves=31,
            ),
        )))

    if _use_spark_cb():
        _candidates.append(("spark_catboost", SparkCatBoostClassifier(
            iterations=_BOOSTER_ROUNDS, depth=6, learningRate=0.05,
            featuresCol="features", labelCol=TARGET_COL, weightCol="weight",
            scalePosWeight=_spw, lossFunction="Logloss",
        )))

    _leader = {"val_ap": -1.0, "name": None}
    for _name, _clf in _candidates:
        try:
            _vap = _train_and_log(_name, _clf, _tr, _va, _te, _spw)
            if _vap > _leader["val_ap"]:
                _r = SPARK_ML_RESULTS[_name]
                _leader = {"name": _name, "val_ap": _vap, "model": _r["model"],
                           "test_ap": _r["test_ap"], "test_pred": _r["test_pred"],
                           "val_pred": _r["val_pred"], "run_id": _r["run_id"]}
        except Exception as _exc:
            print(f"[SKIP] {_name} failed: {_exc}")
            _err = str(_exc).lower()
            if (
                _name.startswith("spark_lgb")
                or _name.startswith("spark_catboost")
                or "connection refused" in _err
                or "py4j" in _err
            ):
                try:
                    ensure_spark_alive(recreate=True)
                except Exception as _re:
                    print(f"[WARN] Spark recreate after {_name} failure: {_re}")

    if _leader["name"] is None:
        raise RuntimeError("All Spark ML candidates failed — check JARs / xgboost / synapseml / catboost_spark")

    CHAMPION_NAME = _leader["name"]
    CHAMPION_MODEL = _leader["model"]
    CATEGORY_RESULTS = {CATEGORY: {
        "champion": CHAMPION_NAME, "champion_model": CHAMPION_MODEL,
        "val_ap": _leader["val_ap"], "test_ap": _leader["test_ap"],
        "champion_run_id": _leader["run_id"],
    }}
    print(f"\nBaseline champion: {CHAMPION_NAME}  val_ap={_leader['val_ap']:.4f}  test_ap={_leader['test_ap']:.4f}")
else:
    print("[CHECKPOINT] Skip CELL 11 training — restored/saved checkpoint.")


In [ ]:
#  CELL 12 : Optuna HPT — Spark XGB / LGB / CatBoost (distributed)
if should_run_training():
    if "_build_stages" not in globals():
        raise RuntimeError("Run CELL 11 first — Spark ML training helpers are defined there.")
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    ensure_spark_alive()
    _NUM_WORKERS, _BOOSTER_ROUNDS = spark_booster_profile()
    _n_workers = _NUM_WORKERS

    CATEGORY = DEVICE_CAT
    cat = CATEGORIES[CATEGORY]
    _tr = cat["train_sdf"].withColumn(
        "weight", F.when(F.col(TARGET_COL) == 1, F.lit(float(cat["scale_pos_weight"]))).otherwise(F.lit(1.0)))
    _va = cat["val_sdf"]
    _te = cat["test_sdf"]
    _OPT_CKPT = CHECKPOINT_FILE.replace("optuna_params", "optuna_spark_params")

    def _spark_optuna_xgb(trial):
        if not _HAS_SPARK_XGB:
            raise optuna.TrialPruned("Spark XGB unavailable")
        clf = SparkXGBClassifier(
            features_col="features", label_col=TARGET_COL, weight_col="weight",
            num_workers=_n_workers, n_estimators=trial.suggest_int("n_estimators", 100, _BOOSTER_ROUNDS),
            max_depth=trial.suggest_int("max_depth", 4, 9),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            subsample=trial.suggest_float("subsample", 0.6, 0.95),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 0.95),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 30),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            eval_metric="aucpr", seed=42,
        )
        model = Pipeline(stages=_build_stages(clf)).fit(_tr)
        return _eval_ap(model, _va)[0]

    def _spark_optuna_lgb(trial):
        if not _use_spark_lgb():
            raise optuna.TrialPruned("Spark LGB unavailable or skipped on local")
        clf = LightGBMClassifier(
            **_spark_lgb_kwargs(
                numIterations=trial.suggest_int("numIterations", 100, _BOOSTER_ROUNDS),
                numLeaves=trial.suggest_int("numLeaves", 20, 200),
                learningRate=trial.suggest_float("learningRate", 0.01, 0.2, log=True),
                baggingFraction=trial.suggest_float("baggingFraction", 0.6, 0.95),
                featureFraction=trial.suggest_float("featureFraction", 0.6, 0.95),
            ),
        )
        model = Pipeline(stages=_build_stages(clf)).fit(_tr)
        return _eval_ap(model, _va)[0]

    def _spark_optuna_cb(trial):
        if not _use_spark_cb():
            raise optuna.TrialPruned("Spark CatBoost unavailable or skipped on local")
        clf = SparkCatBoostClassifier(
            iterations=trial.suggest_int("iterations", 100, _BOOSTER_ROUNDS),
            depth=trial.suggest_int("depth", 4, 9),
            learningRate=trial.suggest_float("learningRate", 0.01, 0.2, log=True),
            l2LeafReg=trial.suggest_float("l2LeafReg", 1.0, 20.0),
            featuresCol="features", labelCol=TARGET_COL, weightCol="weight",
            scalePosWeight=cat["scale_pos_weight"], lossFunction="Logloss",
        )
        model = Pipeline(stages=_build_stages(clf)).fit(_tr)
        return _eval_ap(model, _va)[0]

    _OPT_SPECS = [
        ("spark_xgb_optuna", _spark_optuna_xgb, 40, 600),
    ]
    if _use_spark_lgb():
        _OPT_SPECS.append(("spark_lgb_optuna", _spark_optuna_lgb, 30, 480))
    if _use_spark_cb():
        _OPT_SPECS.append(("spark_catboost_optuna", _spark_optuna_cb, 20, 360))

    for _oname, _obj, _nt, _to in _OPT_SPECS:
        try:
            study = optuna.create_study(direction="maximize", study_name=_oname)
            study.optimize(_obj, n_trials=_nt, timeout=_to, show_progress_bar=False)
            _best = study.best_params
            print(f"\n{_oname}: best val_ap={study.best_value:.4f}  params={_best}")
            # Refit best on train, eval test, log MLflow
            if _oname == "spark_xgb_optuna" and _HAS_SPARK_XGB:
                _best_clf = SparkXGBClassifier(
                    features_col="features", label_col=TARGET_COL, weight_col="weight",
                    num_workers=_n_workers, eval_metric="aucpr", seed=42, **_best)
            elif _oname == "spark_lgb_optuna" and _use_spark_lgb():
                _best_clf = LightGBMClassifier(**_spark_lgb_kwargs(**_best))
            elif _oname == "spark_catboost_optuna" and _use_spark_cb():
                _best_clf = SparkCatBoostClassifier(
                    featuresCol="features", labelCol=TARGET_COL, weightCol="weight",
                    scalePosWeight=cat["scale_pos_weight"], lossFunction="Logloss", **_best)
            else:
                continue
            with mlflow.start_run(run_name=_oname) as run:
                mlflow.set_tags({**COMMON_TAGS, "model_type": _oname, "optuna_trials": _nt})
                mlflow.log_params({**_best, "scale_pos_weight": cat["scale_pos_weight"]})
                _model = Pipeline(stages=_build_stages(_best_clf)).fit(_tr)
                _vap, _vauc, _vp = _eval_ap(_model, _va)
                _tap, _tauc, _tp = _eval_ap(_model, _te)
                mlflow.log_metrics({"val_ap": _vap, "val_auc": _vauc, "test_ap": _tap, "test_auc": _tauc})
                mlflow.spark.log_model(_model, "model")
                SPARK_ML_RESULTS[_oname] = {
                    "model": _model, "run_id": run.info.run_id, "best_params": _best,
                    "val_ap": _vap, "test_ap": _tap, "val_pred": _vp, "test_pred": _tp,
                }
                try:
                    _save_spark_model_to_disk(_oname, _model)
                except Exception as _ckpt_exc:
                    print(f"[WARN] Incremental checkpoint for {_oname} failed: {_ckpt_exc}")
                if _vap > CATEGORY_RESULTS[CATEGORY]["val_ap"]:
                    CHAMPION_NAME = _oname
                    CHAMPION_MODEL = _model
                    CATEGORY_RESULTS[CATEGORY].update({
                        "champion": _oname, "champion_model": _model,
                        "val_ap": _vap, "test_ap": _tap,
                        "champion_run_id": run.info.run_id,
                    })
                    print(f"  >> New champion: {_oname}  val_ap={_vap:.4f}")
        except Exception as _exc:
            print(f"[OPTUNA SKIP] {_oname}: {_exc}")
            _err = str(_exc).lower()
            if (
                "lgb" in _oname
                or "catboost" in _oname
                or "connection refused" in _err
                or "py4j" in _err
            ):
                try:
                    ensure_spark_alive(recreate=True)
                except Exception as _re:
                    print(f"[WARN] Spark recreate after {_oname}: {_re}")

    try:
        joblib.dump({k: SPARK_ML_RESULTS[k].get("best_params") for k in SPARK_ML_RESULTS if "best_params" in SPARK_ML_RESULTS.get(k, {})}, _OPT_CKPT)
    except Exception:
        pass
    print(f"\nChampion after Optuna: {CHAMPION_NAME}")
    save_spark_training_checkpoint()
else:
    print("[CHECKPOINT] Skip CELL 12 Optuna — restored/saved checkpoint.")


In [ ]:
# =============================================================================
#  CELL 12B : Snapshot the trained models before evaluation
# =============================================================================
# On 24-Sep a finished training run's models were deleted before CELL 13 ran, and
# the hour of compute went with them. This copies spark_models and the checkpoint
# meta to a snapshot folder nothing else in the notebook touches. The two most
# recent snapshots are kept.
#
# To restore one: copy its spark_models/ and spark_ckpt_meta.joblib back into
# CKPT_DIR, set FRESH_REBUILD = False in CELL 0B, restart the kernel and Run All.
import os, shutil, datetime as _dtm

if os.path.isdir(SPARK_CKPT_MODELS) and os.listdir(SPARK_CKPT_MODELS):
    _snap_root = CKPT_DIR.rstrip("/") + "_snapshots"
    _snap = os.path.join(_snap_root, f"trained_{_dtm.datetime.now():%Y%m%d_%H%M%S}")
    os.makedirs(_snap, exist_ok=True)
    shutil.copytree(SPARK_CKPT_MODELS, os.path.join(_snap, "spark_models"))
    if os.path.exists(SPARK_CKPT_META):
        shutil.copy2(SPARK_CKPT_META, _snap)
    _all = sorted(d for d in os.listdir(_snap_root) if d.startswith("trained_"))
    for _old in _all[:-2]:
        shutil.rmtree(os.path.join(_snap_root, _old), ignore_errors=True)
    print(f"[snapshot] {_snap}")
    print(f"[snapshot] models: {sorted(os.listdir(os.path.join(_snap, 'spark_models')))}")
else:
    print("[snapshot] no trained models on disk -- nothing to snapshot")


In [ ]:
#  CELL 13 : Spark ML threshold + quality gate + evaluation plots
# >>> Run this cell after CELL 11/12 to see confusion matrix, ROC, PR, calibration, feature importance.
%matplotlib inline
import shutil
from pyspark.ml.functions import vector_to_array
from sklearn.metrics import ConfusionMatrixDisplay

print("=" * 60)
print(f"[DIAG] Evaluation dashboard — {DEVICE_CAT}  champion={CHAMPION_NAME}")
print("=" * 60)

ensure_spark_alive()
DIAG_DIR = os.path.join(CKPT_DIR, "diagnostics")
os.makedirs(DIAG_DIR, exist_ok=True)

CATEGORY = DEVICE_CAT
_qg_floor = CATEGORIES[CATEGORY]["recall_floor"]
_val_pdf = None
_test_pdf = None


def _extract_feature_importances(clf, feature_cols):
    if hasattr(clf, "getFeatureImportances"):
        return np.array(clf.getFeatureImportances(), dtype=float)
    if hasattr(clf, "featureImportances"):
        return np.array(clf.featureImportances.toArray(), dtype=float)
    if hasattr(clf, "get_booster"):
        _score = clf.get_booster().get_score(importance_type="gain")
        return np.array([float(_score.get(f"f{i}", 0.0)) for i in range(len(feature_cols))])
    if hasattr(clf, "getNativeBooster"):
        _score = clf.getNativeBooster().getScore("", "gain")
        return np.array([float(_score.get(str(i), _score.get(f"f{i}", 0.0))) for i in range(len(feature_cols))])
    if hasattr(clf, "coefficients"):
        return np.abs(np.array(clf.coefficients.toArray(), dtype=float))
    return None


try:
    rehydrate_champion_predictions(CATEGORY)
    _val_pred = SPARK_ML_RESULTS[CHAMPION_NAME]["val_pred"]
    _test_pred = SPARK_ML_RESULTS[CHAMPION_NAME]["test_pred"]
    _prob = vector_to_array(F.col("probability"))[1]
    _val_pdf = (
        _val_pred.select(TARGET_COL, _prob.alias("score"))
        .sample(withReplacement=False, fraction=min(1.0, 200_000 / max(_val_pred.count(), 1)), seed=42)
        .toPandas()
    )
    _select_cols = []
    for _c in ["DEVICE_ID", "DEVICE_KEY", "FACILITY_ID", "event_date", "transit_day", TARGET_COL]:
        if _c in _test_pred.columns and _c not in _select_cols:
            _select_cols.append(_c)
    _test_pdf = (
        _test_pred.select(*_select_cols, _prob.alias("score"))
        .toPandas()
    )
except Exception as _pred_exc:
    print(f"[DIAG] Live predictions unavailable ({type(_pred_exc).__name__}) — loading saved test_pdf ...")
    _tp = os.path.join(SPARK_CKPT_EVAL, "test_pdf.parquet")
    if os.path.exists(_tp):
        _test_pdf = pd.read_parquet(_tp)

if _test_pdf is None or not len(_test_pdf):
    raise RuntimeError(
        "No test predictions available. Run CELL 11/12 training first, "
        "or restore a trained/evaluated checkpoint (Cells 1-4 + 5)."
    )

if _val_pdf is not None and len(_val_pdf):
    _y = _val_pdf[TARGET_COL].astype(int).values
    _s = _val_pdf["score"].values
    _prec, _rec, _thr = precision_recall_curve(_y, _s)
    _f2 = (1 + 4) * _prec * _rec / np.maximum(4 * _prec + _rec, 1e-12)
    _f2_i = int(np.nanargmax(_f2))
    opt_thresh = float(_thr[_f2_i]) if _f2_i < len(_thr) else 0.5
    _qg_recall = float(_rec[_f2_i])
    if os.environ.get("PS1_THRESHOLD_RULE", "f2").strip().lower() == "recall_floor":
        # PK 25-Sep: operate AT the recall floor, not the F2 optimum (~0.9 recall).
        # Recall is traded down to the floor and no further; precision, accuracy and FPR
        # improve, AUC does not move. Highest threshold keeping val recall >= the floor.
        _ok = np.where(_rec[:-1] >= _qg_floor)[0]
        if len(_ok):
            _f2_i = int(_ok[-1])
            opt_thresh = float(_thr[_f2_i])
            _qg_recall = float(_rec[_f2_i])
    _val_ap = average_precision_score(_y, _s)
    qg_pass = _qg_recall >= _qg_floor
else:
    _cr = CATEGORY_RESULTS.get(CATEGORY, {})
    opt_thresh = float(_cr.get("opt_thresh", 0.5))
    _qg_recall = float(_cr.get("qg_recall", 0.0))
    qg_pass = bool(_cr.get("qg_pass", False))
    _y = np.array([], dtype=int)
    _s = np.array([], dtype=float)
    _prec, _rec = np.array([]), np.array([])
    _val_ap = float("nan")
    print(f"[DIAG] Using saved threshold={opt_thresh:.4f} (val scores not in session)")

_test_y = _test_pdf[TARGET_COL].astype(int).values
_test_s = _test_pdf["score"].values
_pred_val = (_s >= opt_thresh).astype(int) if len(_s) else np.array([], dtype=int)
_pred_test = (_test_s >= opt_thresh).astype(int)
_champ_ap = average_precision_score(_test_y, _test_s)
_champ_auc = roc_auc_score(_test_y, _test_s)
_test_f1 = f1_score(_test_y, _pred_test)
_test_prec = precision_score(_test_y, _pred_test, zero_division=0)
_test_rec = recall_score(_test_y, _pred_test, zero_division=0)
_tn, _fp, _fn, _tp = confusion_matrix(_test_y, _pred_test, labels=[0, 1]).ravel()


def _expected_calibration_error(y_true, scores, n_bins=10):
    scores = np.clip(scores, 1e-6, 1 - 1e-6)
    ece = 0.0
    edges = np.linspace(0, 1, n_bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (scores >= lo) & (scores < hi if hi < 1 else scores <= hi)
        if not m.any():
            continue
        ece += m.mean() * abs(float(y_true[m].mean()) - float(scores[m].mean()))
    return float(ece)


_ece = _expected_calibration_error(_test_y, _test_s)

CATEGORY_RESULTS[CATEGORY].update({
    "opt_thresh": opt_thresh, "qg_recall": _qg_recall, "qg_pass": qg_pass, "qg_floor": _qg_floor,
    "test_pdf": _test_pdf, "test_ap": float(_champ_ap), "test_auc": float(_champ_auc),
    "test_f1": float(_test_f1), "test_prec": float(_test_prec), "test_rec": float(_test_rec),
    "confusion": {"tn": int(_tn), "fp": int(_fp), "fn": int(_fn), "tp": int(_tp)},
})
print(f"Threshold={opt_thresh:.4f}  rule={os.environ.get('PS1_THRESHOLD_RULE', 'f2')}  "
      f"val recall={_qg_recall:.4f}  QG={'PASS' if qg_pass else 'FAIL'}")
print(f"Test AP={_champ_ap:.4f}  AUC={_champ_auc:.4f}  F1={_test_f1:.4f}  Prec={_test_prec:.4f}  Rec={_test_rec:.4f}  "
      f"Acc={(_tn + _tp) / max(1, _tn + _fp + _fn + _tp):.4f}  FPR={_fp / max(1, _fp + _tn):.4f}")
print(f"Confusion TN={_tn} FP={_fp} FN={_fn} TP={_tp}  ECE={_ece:.4f}")

_prefix = f"{DEVICE_CAT.lower()}"
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(f"{DEVICE_CAT} champion={CHAMPION_NAME}  threshold={opt_thresh:.3f}", fontsize=12)

ConfusionMatrixDisplay(
    confusion_matrix(_test_y, _pred_test, labels=[0, 1]), display_labels=["neg", "pos"]
).plot(ax=axes[0, 0], cmap="Blues", colorbar=False)
axes[0, 0].set_title("Confusion matrix (test)")

if len(_y):
    ConfusionMatrixDisplay(
        confusion_matrix(_y, _pred_val, labels=[0, 1]), display_labels=["neg", "pos"]
    ).plot(ax=axes[0, 1], cmap="Oranges", colorbar=False)
    axes[0, 1].set_title("Confusion matrix (val)")
else:
    axes[0, 1].text(0.5, 0.5, "Val CM\n(re-run with live preds)", ha="center", va="center", transform=axes[0, 1].transAxes)
    axes[0, 1].set_axis_off()

_fpr, _tpr, _ = roc_curve(_test_y, _test_s)
axes[0, 2].plot(_fpr, _tpr, label=f"AUC={_champ_auc:.3f}")
axes[0, 2].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[0, 2].set_xlabel("FPR")
axes[0, 2].set_ylabel("TPR")
axes[0, 2].set_title("ROC curve (test)")
axes[0, 2].legend(loc="lower right")

if len(_rec):
    axes[1, 0].plot(_rec, _prec, color="#E53935")
    axes[1, 0].axvline(_qg_recall, color="gray", ls="--", lw=0.8, label=f"val F2 recall={_qg_recall:.2f}")
    axes[1, 0].set_title(f"PR curve (val)  AP={_val_ap:.3f}")
    axes[1, 0].legend(loc="upper right")
else:
    _tprec, _trec, _ = precision_recall_curve(_test_y, _test_s)
    axes[1, 0].plot(_trec, _tprec, color="#E53935")
    axes[1, 0].set_title(f"PR curve (test)  AP={_champ_ap:.3f}")
axes[1, 0].set_xlabel("Recall")
axes[1, 0].set_ylabel("Precision")

_bin_edges = np.linspace(0, 1, 11)
_bin_centers = 0.5 * (_bin_edges[:-1] + _bin_edges[1:])
_bin_acc = []
for lo, hi in zip(_bin_edges[:-1], _bin_edges[1:]):
    m = (_test_s >= lo) & (_test_s < hi if hi < 1 else _test_s <= hi)
    _bin_acc.append(float(_test_y[m].mean()) if m.any() else 0.0)
axes[1, 1].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[1, 1].bar(_bin_centers, _bin_acc, width=0.08, alpha=0.7, color="#3949AB")
axes[1, 1].set_xlabel("Mean predicted score")
axes[1, 1].set_ylabel("Observed positive rate")
axes[1, 1].set_title(f"Calibration curve (test)  ECE={_ece:.3f}")

_imp = None
try:
    _pipe = SPARK_ML_RESULTS[CHAMPION_NAME]["model"]
    _clf = _pipe.stages[-1]
    _imp = _extract_feature_importances(_clf, FEATURE_COLS)
except Exception as _ie:
    print(f"[DIAG] Feature importance unavailable: {_ie}")

if _imp is not None and len(_imp) == len(FEATURE_COLS):
    _imp_top = pd.Series(_imp, index=FEATURE_COLS).sort_values(ascending=False).head(15)
    _imp_top.iloc[::-1].plot(kind="barh", ax=axes[1, 2], color="#00897B")
    axes[1, 2].set_title("Top-15 feature importances")
    _imp_csv = os.path.join(DIAG_DIR, f"{_prefix}_feature_importance.csv")
    _imp_top.to_csv(_imp_csv)
    print(f"[DIAG] Feature importance CSV -> {_imp_csv}")
else:
    axes[1, 2].text(0.5, 0.5, "Feature importance N/A\n(Spark model)", ha="center", va="center", transform=axes[1, 2].transAxes)
    axes[1, 2].set_axis_off()

plt.tight_layout()
_diag_png = os.path.join(DIAG_DIR, f"{_prefix}_eval_dashboard.png")
plt.savefig(_diag_png, dpi=120, bbox_inches="tight")
if "OUT_DIR" in globals() and os.path.isdir(OUT_DIR):
    _out_copy = os.path.join(OUT_DIR, f"{_prefix}_eval_dashboard.png")
    try:
        shutil.copy(_diag_png, _out_copy)
        print(f"[DIAG] Also copied -> {_out_copy}")
    except Exception:
        pass
plt.show()
print(f"[DIAG] Dashboard saved -> {_diag_png}")
print("[DIAG] Plots: confusion (test/val), ROC, PR, calibration, feature importance")

save_spark_eval_checkpoint()


In [ ]:
#  CELL 14 : Legacy clean-label stub (disabled — Feature Store is CELL 15)
# Planned events (software resets, depot maintenance, scheduled PM) inflate the
# positive rate and teach the model to predict operator-initiated actions rather
# than unplanned failures. Recompute will_fail_3d_clean when event_type_id available.
# Ported from PS1_Evaluation_Fix.ipynb Cell 4b.
_USE_CLEAN_LABEL = False  # SLA clean-label path disabled for OOS target
if False and "event_type_id" in df.columns:
    print("event_type_id found — recomputing will_fail_3d_clean ...")
    _planned = {
        "TVM":  set(TVM_PLANNED_EVENT_TYPES),
        "GATE": set(GATE_PLANNED_EVENT_TYPES),
    }
    df["_unplanned"] = 1
    for _cat in [DEVICE_CAT]:
        _mask = df["mars_device_category"] == _cat
        df.loc[_mask & df["event_type_id"].isin(_planned[_cat]), "_unplanned"] = 0
        _zeroed = int((_mask & (df["_unplanned"] == 0)).sum())
        print(f"  {_cat:<6}: {_zeroed:,} planned rows zeroed ({len(_planned[_cat])} event types)")
    df = df.sort_values(["DEVICE_ID", "event_date"]).copy()
    df["will_fail_3d_clean"] = (
        df.groupby("DEVICE_ID")["_unplanned"]
        .transform(lambda x: x.shift(-1).rolling(3, min_periods=1).max().fillna(0))
        .astype(int)
    )
    df.drop(columns=["_unplanned"], inplace=True)
    _USE_CLEAN_LABEL = True
    print("\n[Clean label positive rate]")
    for _cat in [DEVICE_CAT]:
        _m = df[df["mars_device_category"] == _cat]
        _orig  = _m[TARGET].mean()
        _clean = _m["will_fail_3d_clean"].mean()
        print(f"  {_cat:<6}: original={_orig:.3%}  clean={_clean:.3%}  "
              f"delta={_orig - _clean:.3%}  (planned events excluded)")
    # Update CATEGORIES splits to use clean label
    _target_col = "will_fail_3d_clean"
    for _cat in [DEVICE_CAT]:
        for _split in ["train_df", "val_df", "test_df"]:
            _df_s = CATEGORIES[_cat][_split]
            if _target_col in _df_s.columns:
                CATEGORIES[_cat][f"y_{_split.replace('_df','')}_clean"] = _df_s[_target_col].values
    print("\n[ACTIVE] will_fail_3d_clean applied to CATEGORIES splits.")
else:
    print(f"[Skip] Clean SLA label not applied — active target is {TARGET_COL} (hardware OOS Set).")
    print(f"  VALIDATOR planned types ({len(VALIDATOR_PLANNED_EVENT_TYPES)}): {VALIDATOR_PLANNED_EVENT_TYPES}")


In [ ]:
#  CELL 15 : Feature Store -- create group + seed ONLINE store (Spark)
# The Feature Group is PRODUCTION and this cell ingests TARGET_COL, not just the
# features -- so an experimental label would land in the online store and the offline
# S3 copy under the production group name. Gate it on the same expression that blocks
# CELL 21 and CELL 24. Skip rather than raise, so the evaluation cells still run and
# the CELL 21 guard's "cells 1-20 give the metrics" stays true.
if _PS1_CONFIG_NON_DEFAULT:
    print('[EXPERIMENT] Feature Store ingest SKIPPED -- the label configuration is '
          'non-default, and CELL 15 writes TARGET_COL into the production Feature '
          'Group. Nothing was ingested. Metrics in the cells below are unaffected.')
elif not ENABLE_FEATURE_STORE:
    print('Feature Store disabled -- skipping CELL 15.')
else:
    from sagemaker.feature_store.feature_group import FeatureGroup
    from sagemaker.feature_store.feature_definition import FeatureDefinition, FeatureTypeEnum
    import time

    OFFLINE_URI = f"s3://{ARTIFACT_BUCKET}/feature-store/{FS_GROUP_NAME}"

    if "df_ml_spark" not in globals():
        raise RuntimeError("df_ml_spark missing — run CELL 6-9 (ETL) first.")

    _fs_cols = [
        c for c in FEATURE_COLS + [TARGET_COL, "DEVICE_ID", "event_date"]
        if c in df_ml_spark.columns
    ]
    if not _fs_cols:
        raise RuntimeError("No Feature Store columns found on df_ml_spark.")

    from pyspark.sql.window import Window as _FsW

    _fsw = _FsW.partitionBy("DEVICE_ID").orderBy(F.col("event_date").desc())
    fs_full = (
        df_ml_spark.select(*_fs_cols)
        .withColumn("_rn", F.row_number().over(_fsw))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
        .toPandas()
    )
    fs_full["DEVICE_ID"] = fs_full["DEVICE_ID"].astype(str)
    if "event_date" not in fs_full.columns and "transit_day" in fs_full.columns:
        fs_full["event_date"] = fs_full["transit_day"]
    fs_full["event_date"] = pd.to_datetime(fs_full["event_date"], errors="coerce", utc=True)
    fs_latest = (
        fs_full.sort_values("event_date")
        .groupby("DEVICE_ID", as_index=False)
        .tail(1)
        .reset_index(drop=True)
    )
    _bad_dates = fs_latest["event_date"].isna()
    if _bad_dates.any():
        print(f"[FS] {_bad_dates.sum()} row(s) with unparseable event_date — using UTC midnight")
        fs_latest.loc[_bad_dates, "event_date"] = pd.Timestamp("1970-01-01", tz="UTC")
    fs_latest["event_time"] = fs_latest["event_date"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    fs_latest = fs_latest.drop(columns=["event_date"])
    num_cols = fs_latest.select_dtypes(include="number").columns
    fs_latest[num_cols] = fs_latest[num_cols].fillna(0)
    print(f"Candidate columns for Feature Store: {len(fs_latest.columns)}")

    def fs_type(dt):
        if pd.api.types.is_float_dtype(dt):
            return FeatureTypeEnum.FRACTIONAL
        if pd.api.types.is_integer_dtype(dt):
            return FeatureTypeEnum.INTEGRAL
        return FeatureTypeEnum.STRING

    def _make_fg(cols_df):
        return FeatureGroup(
            name=FS_GROUP_NAME,
            sagemaker_session=SM_SESSION,
            feature_definitions=[
                FeatureDefinition(c, fs_type(cols_df[c].dtype)) for c in cols_df.columns
            ],
        )

    def _fg_status(fg):
        try:
            return fg.describe().get("FeatureGroupStatus")
        except Exception:
            return None

    def _fg_registered_cols(fg):
        try:
            return {
                f["FeatureName"]: f["FeatureType"]
                for f in fg.describe().get("FeatureDefinitions", [])
            }
        except Exception:
            return {}

    _probe = _make_fg(fs_latest)
    st = _fg_status(_probe)
    if st in ("Created", "CreateFailed"):
        existing = _fg_registered_cols(_probe)
        _added = set(fs_latest.columns) - set(existing)
        _type_mismatch = {
            c: (fs_type(fs_latest[c].dtype).value, existing[c])
            for c in fs_latest.columns
            if c in existing and fs_type(fs_latest[c].dtype).value != existing[c]
        }
        if st == "CreateFailed" or _type_mismatch:
            if _type_mismatch:
                print(f"  Type mismatch vs existing schema: {_type_mismatch}")
            print(f"Deleting stale Feature Group (status={st})...")
            _probe.delete()
            from botocore.exceptions import ClientError as _FS_DelClientError

            _MAX_DELETE_WAIT = 30
            for _wait_i in range(_MAX_DELETE_WAIT):
                try:
                    _make_fg(fs_latest).describe()
                    _still_exists = True
                except _FS_DelClientError as _del_ce:
                    _still_exists = (
                        _del_ce.response.get("Error", {}).get("Code") != "ResourceNotFound"
                    )
                except Exception:
                    _still_exists = True
                if not _still_exists:
                    print("  Deleted.")
                    break
                print(f"  Still deleting... ({_wait_i + 1}/{_MAX_DELETE_WAIT})")
                time.sleep(10)
            time.sleep(20)
            st = None
        elif _added:
            print(f"  New columns not in existing schema: {sorted(_added)}")
            _probe.update(
                feature_additions=[
                    FeatureDefinition(c, fs_type(fs_latest[c].dtype)) for c in sorted(_added)
                ]
            )
            while _fg_status(_probe) != "Created":
                time.sleep(10)
                print("  status:", _fg_status(_probe))
            print("  Update complete -- reusing existing Feature Group.")
        else:
            print(
                f"Schema OK ({len(existing)} cols, names + types) -- reusing existing Feature Group."
            )

    if st is None:
        feature_group = _make_fg(fs_latest)
        from botocore.exceptions import ClientError as _FS_ClientError

        _MAX_CREATE_RETRIES = 15
        _CREATE_RETRY_WAIT_S = 30
        for _create_attempt in range(_MAX_CREATE_RETRIES):
            try:
                feature_group.create(
                    s3_uri=OFFLINE_URI,
                    record_identifier_name="DEVICE_ID",
                    event_time_feature_name="event_time",
                    role_arn=ROLE,
                    enable_online_store=True,
                    description="PS1 daily device features -- Chicago Ventra",
                )
                break
            except _FS_ClientError as _fs_ce:
                _is_in_use = _fs_ce.response.get("Error", {}).get("Code") == "ResourceInUse"
                if _is_in_use and _create_attempt < _MAX_CREATE_RETRIES - 1:
                    print(
                        f"  create() hit ResourceInUse -- retry {_create_attempt + 1} "
                        f"in {_CREATE_RETRY_WAIT_S}s..."
                    )
                    time.sleep(_CREATE_RETRY_WAIT_S)
                else:
                    raise
        print("Creating Feature Group...")
        while _fg_status(feature_group) != "Created":
            time.sleep(15)
            print("  status:", _fg_status(feature_group))
        print("Waiting 60s for schema propagation...")
        time.sleep(60)
    else:
        feature_group = _probe

    registered = _fg_registered_cols(feature_group)
    unregistered = set(fs_latest.columns) - set(registered)
    if unregistered:
        print(f"NOTE: {len(unregistered)} columns not registered by SDK:")
        print(f"  {sorted(unregistered)}")
        fs_ingest = fs_latest[[c for c in fs_latest.columns if c in registered]].copy()
    else:
        fs_ingest = fs_latest
        print(f"All {len(registered)} columns registered -- ingesting full set.")

    print(f"Feature Group ready: {FS_GROUP_NAME}  ({len(registered)} features)")

    from sagemaker.feature_store.feature_group import IngestionError

    try:
        feature_group.ingest(
            data_frame=fs_ingest.reset_index(drop=True), max_workers=4, wait=True
        )
    except IngestionError as _ie:
        _n_failed = len(_ie.failed_rows)
        print(f"[FS] Ingestion failed for {_n_failed}/{len(fs_ingest)} rows.")
        _step = max(1, _n_failed // 5)
        _sample_idx = sorted(set(_ie.failed_rows[0::_step][:6]))
        _rt = SM_SESSION.boto_session.client("sagemaker-featurestore-runtime")
        _real_errors = {}
        for _fi in _sample_idx:
            _row = fs_ingest.iloc[[_fi]].reset_index(drop=True)
            _record = [
                {"FeatureName": c, "ValueAsString": str(_row.iloc[0][c])} for c in _row.columns
            ]
            try:
                _rt.put_record(FeatureGroupName=FS_GROUP_NAME, Record=_record)
            except Exception as _real_e:
                _real_errors[_fi] = f"{type(_real_e).__name__}: {_real_e}"
        if _real_errors:
            for _fi, _err in _real_errors.items():
                print(f"  row {_fi}: {_err}")
            raise _ie
        time.sleep(30)
        _retry_df = fs_ingest.iloc[_ie.failed_rows].reset_index(drop=True)
        feature_group.ingest(data_frame=_retry_df, max_workers=1, wait=True)

    sample_id = fs_ingest["DEVICE_ID"].iloc[0]
    rec = feature_group.get_record(record_identifier_value_as_string=sample_id)
    print(
        f"Ingested {len(fs_ingest):,} records | online sample ({sample_id}):",
        {f["FeatureName"]: f["ValueAsString"] for f in (rec or [])[:4]},
    )
    globals()["feature_group"] = feature_group


In [ ]:
#  CELL 16 : Feature Store -- online store verify (per category)
# Creation and ingestion are in CELL 15.
if not ENABLE_FEATURE_STORE:
    print("Feature Store disabled — skipping CELL 16.")
elif "feature_group" not in globals():
    print("[FS] feature_group missing — run CELL 15 first.")
else:
    for cat_name in [DEVICE_CAT]:
        _sample_pdf = (
            df_ml_spark.filter(F.col("mars_device_category") == cat_name)
            .select("DEVICE_ID")
            .distinct()
            .limit(1)
            .toPandas()
        )
        sample_ids = (
            _sample_pdf["DEVICE_ID"].astype(str).tolist() if len(_sample_pdf) else []
        )
        if sample_ids:
            rec = feature_group.get_record(record_identifier_value_as_string=sample_ids[0])
            print(
                f"[{cat_name}] online store ({sample_ids[0]}):",
                {f["FeatureName"]: f["ValueAsString"] for f in (rec or [])[:4]},
            )
        else:
            print(f"[{cat_name}] no devices found for spot-check")
    print("Feature Store verification complete.")


In [ ]:
#  CELL 17 : Helper functions (Spark ML evaluation utilities)
def make_common_tags(category):
    return {
        "ps": "PS1",
        "category": category,
        "target": TARGET_COL,
        "city": "chicago",
        "gold_version": str(GOLD_VERSION),
        "grain": "device_day",
        "architecture": "CUBIC-MARS-AWS-sparkml-v1",
        "pipeline": "spark_ml_boosters",
    }

def threshold_sweep(y_true, y_prob, thresholds=None):
    """Precision/Recall/F1 table across thresholds (numpy arrays)."""
    if thresholds is None:
        thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
    rows = []
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        tp = int(((pred == 1) & (y_true == 1)).sum())
        fp = int(((pred == 1) & (y_true == 0)).sum())
        fn = int(((pred == 0) & (y_true == 1)).sum())
        prec = tp / max(tp + fp, 1)
        rec = tp / max(tp + fn, 1)
        f1 = 2 * prec * rec / max(prec + rec, 1e-9)
        rows.append({
            "threshold": t,
            "Recall": round(rec, 4),
            "Precision": round(prec, 4),
            "F1": round(f1, 4),
            "TP": tp, "FP": fp, "FN": fn,
        })
    return pd.DataFrame(rows)

def print_ranking_table(label, y_true, y_prob, top_ks=(0.01, 0.05, 0.10)):
    y_true = np.asarray(y_true).astype(float)
    y_prob = np.asarray(y_prob).astype(float)
    base_rate = float(y_true.mean())
    order = np.argsort(y_prob)[::-1]
    print(f"\n{label} — Ranking metrics (base rate = {base_rate:.3%})")
    print(f"  {'K':<8}  {'Prec@K':>8}  {'Rec@K':>8}  {'Lift@K':>8}")
    for k in top_ks:
        n_k = max(int(len(y_true) * k), 1)
        top_y = y_true[order[:n_k]]
        prec = top_y.mean()
        rec = top_y.sum() / max(y_true.sum(), 1)
        lift = prec / max(base_rate, 1e-9)
        print(f"  Top {k:.0%}  {prec:>8.4f}  {rec:>8.4f}  {lift:>7.2f}x")

print("Spark ML helper functions loaded.")


In [ ]:
#  CELL 18 : MLflow leaderboard -- search_runs per experiment (ranked by test_ap)
ensure_mlflow_tracking()
_today = pd.Timestamp.today().strftime("%Y-%m-%d")
for CATEGORY in [DEVICE_CAT]:
    cat = CATEGORIES[CATEGORY]
    ensure_ps1_experiment(cat['experiment_name'])
    _base_filter = "tags.model_type != 'shap_analysis' AND tags.model_type != 'xgboost_multiclass'"
    runs = mlflow.search_runs(
        experiment_names=[cat['experiment_name']],
        filter_string=f"{_base_filter} AND tags.run_date = '{_today}'",
        order_by=['metrics.test_ap DESC'])
    print(f'\n {CATEGORY} [{cat["experiment_name"]}] leaderboard — today ({_today}), ranked by test_ap ')
    if runs.empty:
        print(f"  [INFO] No runs logged today ({_today}). Re-run CELL 11 → 12 → 13 to refresh.")
        continue
    # Deduplicate: keep latest run per model name (handles notebook re-runs).
    if 'tags.mlflow.runName' in runs.columns and 'start_time' in runs.columns:
        runs = (runs.sort_values('start_time', ascending=False)
                    .drop_duplicates('tags.mlflow.runName')
                    .sort_values('metrics.test_ap', ascending=False, ignore_index=True))
    cols = ['tags.mlflow.runName', 'tags.run_date', 'metrics.test_auc', 'metrics.test_ap',
            'metrics.test_f1', 'metrics.test_prec', 'metrics.test_rec',
            'metrics.train_auc', 'metrics.val_auc']
    available = [c for c in cols if c in runs.columns]
    print(runs[available].head(10).to_string(index=False))


In [ ]:
#  CELL 19 : SHAP — Spark tree champion (XGB/LGB/CatBoost)
from pyspark.ml.functions import vector_to_array

_shap_imp_all = {}
CATEGORY = DEVICE_CAT

# Prefer Optuna-tuned tree boosters for SHAP
_shap_key = CHAMPION_NAME
for _pref in ("spark_xgb_optuna", "spark_lgb_optuna", "spark_catboost_optuna", "spark_xgb", "spark_lgb", "spark_catboost"):
    if _pref in SPARK_ML_RESULTS:
        _shap_key = _pref
        break

if _shap_key not in SPARK_ML_RESULTS or SPARK_ML_RESULTS[_shap_key].get("model") is None:
    print(f"[SHAP] {_shap_key} not available — run CELL 11/12 or restore checkpoint.")
else:
    _pipe = SPARK_ML_RESULTS[_shap_key]["model"]
    _clf = _pipe.stages[-1]
    _test_pred = SPARK_ML_RESULTS[_shap_key]["test_pred"]
    _n_shap = 2000

    # Build pandas sample from Spark features vector
    _arr = vector_to_array(F.col("features"))
    _feat_cols = [_arr[i].alias(FEATURE_COLS[i]) for i in range(len(FEATURE_COLS))]
    _prob_expr = vector_to_array(F.col("probability"))[1]
    _pdf = (
        _test_pred
        .select(TARGET_COL, _prob_expr.alias("score"), *_feat_cols)
        .limit(_n_shap)
        .toPandas()
    )
    X_shap = _pdf[FEATURE_COLS].astype(float)
    y_shap = _pdf[TARGET_COL].astype(int).values
    scores = _pdf["score"].values

    _shap_values = None
    try:
        if "xgb" in _shap_key and hasattr(_clf, "get_booster"):
            import xgboost as _xgb_shap
            _bst = _clf.get_booster()
            _dm = _xgb_shap.DMatrix(X_shap.values, feature_names=FEATURE_COLS)
            _contribs = _bst.predict(_dm, pred_contribs=True)
            _shap_values = shap.Explanation(
                values=_contribs[:, :-1], base_values=_contribs[:, -1],
                data=X_shap.values, feature_names=FEATURE_COLS,
            )
        elif "lgb" in _shap_key and hasattr(_clf, "getFeatureImportances"):
            _imp = np.array(_clf.getFeatureImportances())
            _shap_imp_all[CATEGORY] = pd.Series(_imp, index=FEATURE_COLS).sort_values(ascending=False)
            print(f"[SHAP] Spark LGB feature importances (global) — top: {_shap_imp_all[CATEGORY].head(5).to_dict()}")
        elif "catboost" in _shap_key and hasattr(_clf, "getFeatureImportance"):
            _imp = np.array(_clf.getFeatureImportance())
            _shap_imp_all[CATEGORY] = pd.Series(_imp, index=FEATURE_COLS).sort_values(ascending=False)
            print(f"[SHAP] Spark CatBoost feature importances — top: {_shap_imp_all[CATEGORY].head(5).to_dict()}")
    except Exception as _e:
        print(f"[SHAP] Native booster path failed: {_e}")

    if _shap_values is not None:
        _mean_abs = pd.Series(np.abs(_shap_values.values).mean(0), index=FEATURE_COLS).sort_values(ascending=False)
        _shap_imp_all[CATEGORY] = _mean_abs
        print(f"\nSHAP (Spark XGB pred_contribs) — {CATEGORY}  n={len(X_shap)}")
        print(_mean_abs.head(15).to_string())
        try:
            shap.plots.bar(_shap_values, max_display=15, show=False)
            plt.tight_layout(); plt.show()
        except Exception:
            _mean_abs.head(15).plot(kind="barh", figsize=(8, 5), title=f"Mean |SHAP| — {_shap_key}")
            plt.tight_layout(); plt.show()
    elif CATEGORY not in _shap_imp_all:
        # Fallback: Spark RF feature importances if tree model exposes it
        try:
            if hasattr(_clf, "featureImportances"):
                _imp = _clf.featureImportances.toArray()
                _shap_imp_all[CATEGORY] = pd.Series(_imp, index=FEATURE_COLS).sort_values(ascending=False)
                print(_shap_imp_all[CATEGORY].head(15).to_string())
        except Exception as _e2:
            print(f"[SHAP] No importances available: {_e2}")

    CATEGORY_RESULTS[CATEGORY]["shap_key"] = _shap_key
    if CATEGORY in _shap_imp_all:
        CATEGORY_RESULTS[CATEGORY]["shap_top3"] = _shap_imp_all[CATEGORY].head(3).to_dict()


In [ ]:

def _ranking_at_k(y_true, y_prob, k_frac=0.10):
    _n = len(y_true)
    _k = max(1, int(_n * k_frac))
    _idx = np.argsort(y_prob)[::-1][: _k]
    _prec = float(y_true[_idx].sum()) / _k
    _rec = float(y_true[_idx].sum()) / max(float(y_true.sum()), 1)
    return _prec, _rec

#  CELL 20 : Ranking Metrics + Monthly Drift Check (Spark champion scores)
for CATEGORY in [DEVICE_CAT]:
    _res = CATEGORY_RESULTS[CATEGORY]
    _test_pdf = _res.get("test_pdf")
    if _test_pdf is None or _test_pdf.empty:
        print(f"[{CATEGORY}] test_pdf missing — run CELL 13 first.")
        continue
    y_test = _test_pdf[TARGET_COL].astype(int).values
    proba = _test_pdf["score"].astype(float).values
    opt_thresh = _res.get("opt_thresh", 0.5)

    _n = len(y_test)
    _sorted_idx = np.argsort(proba)[::-1]
    _k = max(1, int(_n * 0.10))
    _hits_k = y_test[_sorted_idx[:_k]].sum()
    _total_pos = y_test.sum()
    _prec_k = float(_hits_k) / _k
    _rec_k = float(_hits_k) / max(float(_total_pos), 1)
    _lift_k = _prec_k / max(float(y_test.mean()), 1e-9)
    try:
        from sklearn.metrics import ndcg_score as _ndcg
        _ndcg_k = float(_ndcg(y_test.reshape(1, -1), proba.reshape(1, -1), k=_k))
    except Exception:
        _ndcg_k = float("nan")
    _ap_score = float(average_precision_score(y_test, proba))

    print(f"\n[{CATEGORY}] Ranking metrics (top {_k:,} of {_n:,}, {_k/_n:.0%} coverage)")
    print(f"  Precision@{_k:,} = {_prec_k:.4f}   Recall@{_k:,} = {_rec_k:.4f}")
    print(f"  Lift@{_k:,}      = {_lift_k:.2f}x    MAP = {_ap_score:.4f}   NDCG@k = {_ndcg_k:.4f}")
    if "print_ranking_table" in globals():
        print_ranking_table(f"{CATEGORY} test set", y_test, proba)
    else:
        _p10, _r10 = _ranking_at_k(y_test, proba, 0.10)
        print(f"[{CATEGORY}] top-10% precision={_p10:.4f} recall={_r10:.4f}")

    _res.update({"prec_k": _prec_k, "rec_k": _rec_k, "lift_k": _lift_k, "ndcg_k": _ndcg_k, "map": _ap_score})

    if "event_date" in _test_pdf.columns:
        _df_d = _test_pdf.copy()
        _df_d["_month"] = pd.to_datetime(_df_d["event_date"]).dt.to_period("M")
        _df_d["_proba"] = proba
        _df_d["_actual"] = y_test
        _drift_rows = []
        for _mo, _grp in _df_d.groupby("_month"):
            _n_pos = int(_grp["_actual"].sum())
            if _n_pos < 5:
                continue
            try:
                _auc = roc_auc_score(_grp["_actual"], _grp["_proba"])
                _ap = average_precision_score(_grp["_actual"], _grp["_proba"])
                _drift_rows.append({"month": str(_mo), "n": len(_grp), "pos": _n_pos, "auc": _auc, "ap": _ap})
            except Exception:
                pass
        if _drift_rows:
            print(f"\n[{CATEGORY}] Monthly drift — AUC/AP on test set (min 5 positives)")
            print(f"  {'Month':<12} {'N':>7} {'Pos':>5} {'AUC':>8} {'AP':>8}")
            print("  " + "-" * 46)
            _aucs = [r["auc"] for r in _drift_rows]
            for r in _drift_rows:
                _flag = "  WARN" if r["auc"] < np.mean(_aucs) - 0.05 else ""
                print(f"  {r['month']:<12} {r['n']:>7,} {r['pos']:>5} {r['auc']:>8.4f} {r['ap']:>8.4f}{_flag}")
            _res["drift_monthly"] = _drift_rows

    _pred_out = _test_pdf.copy()
    _pred_out["ps1_fail_prob"] = proba
    _pred_out["ps1_predicted"] = (proba >= opt_thresh).astype(int)
    _pred_out["threshold_used"] = opt_thresh
    if "transit_day" not in _pred_out.columns and "event_date" in _pred_out.columns:
        _pred_out["transit_day"] = _pred_out["event_date"]
    if "DEVICE_KEY" not in _pred_out.columns and "df_ml_spark" in globals():
        try:
            _dk = (
                df_ml_spark.select("DEVICE_ID", "DEVICE_KEY").distinct()
                .toPandas()
            )
            _pred_out = _pred_out.merge(_dk, on="DEVICE_ID", how="left")
        except Exception as _dk_exc:
            print(f"[WARN] DEVICE_KEY join skipped: {_dk_exc}")
    if "DEVICE_KEY" not in _pred_out.columns and "DEVICE_ID" in _pred_out.columns:
        _pred_out["DEVICE_KEY"] = _pred_out["DEVICE_ID"].astype(str)
    _res["predictions"] = _pred_out
    _res["y_test"] = pd.Series(y_test)
    print(f"[{CATEGORY}] Stored {len(_pred_out):,} test predictions for cross-wired output.")


In [ ]:
if _PS1_CONFIG_NON_DEFAULT:
    raise RuntimeError(
        "[EXPERIMENT] publishing is blocked because the label configuration is "
        "non-default. Cells 1-20 give the metrics; CELL 21 (MLflow registration) "
        "and CELL 24 (cross-wire S3 export) would publish an experimental label "
        "under the production name. Unset the PS1_* label vars to publish."
    )

#  CELL 21 : MLflow Model Registration (Spark ML champion via mlflow.spark)
ensure_mlflow_tracking()
apply_ps1_resource_names(CATEGORIES)
import json as _json_mod
from botocore.exceptions import ClientError
from mlflow.tracking import MlflowClient

_mlflow_client = MlflowClient()

for CATEGORY in [DEVICE_CAT]:
    EXPERIMENT_NAME = CATEGORIES[CATEGORY]["experiment_name"]
    MLFLOW_MODEL_NAME = CATEGORIES[CATEGORY]["model_name"]
    MODEL_REGISTRY_NAME = CATEGORIES[CATEGORY]["registry_name"]

    _res = CATEGORY_RESULTS[CATEGORY]
    opt_thresh = _res.get("opt_thresh", 0.5)
    qg_pass = _res.get("qg_pass", False)
    qg_recall = _res.get("qg_recall", 0.0)
    qg_floor = CATEGORIES[CATEGORY].get("recall_floor", QUALITY_RECALL_FLOOR.get(CATEGORY, 0.70))

    print(f"\n{'='*60}")
    print(f"[{CATEGORY}] MLflow Spark Registration")
    print(f"  Quality gate: {'PASS' if qg_pass else 'FAIL'}  recall={qg_recall:.4f}  floor={qg_floor}")
    print(f"  Champion: {CHAMPION_NAME}")
    print(f"{'='*60}")

    _champ_entry = SPARK_ML_RESULTS.get(CHAMPION_NAME, {})
    _champ_run_id = _champ_entry.get("run_id")
    if not _champ_run_id:
        _cr = _res.get("champion_run")
        if _cr is not None:
            _champ_run_id = getattr(getattr(_cr, "info", None), "run_id", None)

    if not _champ_run_id:
        _exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
        if _exp is None:
            print(f"  [WARN] Experiment '{EXPERIMENT_NAME}' not found — skipping")
            continue
        _today = pd.Timestamp.today().strftime("%Y-%m-%d")
        _runs = _mlflow_client.search_runs(
            experiment_ids=[_exp.experiment_id],
            filter_string=f"tags.run_date = '{_today}'",
            order_by=["metrics.test_ap DESC"],
            max_results=20,
        )
        if not _runs:
            _runs = _mlflow_client.search_runs(
                experiment_ids=[_exp.experiment_id],
                order_by=["metrics.test_ap DESC"],
                max_results=20,
            )
        if _runs:
            _champ_run_id = _runs[0].info.run_id

    if not _champ_run_id:
        print("  [WARN] No champion run_id — skipping registration")
        continue

    _champ_ap = float(_champ_entry.get("test_ap", _res.get("test_ap", 0.0)))
    _model_uri = f"runs:/{_champ_run_id}/model"
    print(f"  Run ID     : {_champ_run_id[:8]}...")
    print(f"  Model URI  : {_model_uri}  (spark flavor)")
    print(f"  test_ap    : {_champ_ap:.4f}")

    _new_version = None
    try:
        _mv = mlflow.register_model(model_uri=_model_uri, name=MLFLOW_MODEL_NAME)
        _new_version = _mv.version
        print(f"  Registered : {MLFLOW_MODEL_NAME} v{_new_version}")

        _tag_dict = {
            "champion_run": _champ_run_id,
            "model_type": CHAMPION_NAME,
            "flavor": "spark",
            "test_ap": f"{_champ_ap:.4f}",
            "active_threshold": f"{opt_thresh:.4f}",
            "quality_gate": "PASS" if qg_pass else "FAIL",
            "recall": f"{qg_recall:.4f}",
            "target": TARGET_COL,
        }
        for _tk, _tv in _tag_dict.items():
            _mlflow_client.set_model_version_tag(MLFLOW_MODEL_NAME, _new_version, _tk, _tv)

        for _ev in _mlflow_client.search_model_versions(f"name='{MLFLOW_MODEL_NAME}'"):
            if _ev.current_stage == "Staging" and _ev.version != _new_version:
                _mlflow_client.transition_model_version_stage(MLFLOW_MODEL_NAME, _ev.version, "Archived")
                print(f"  Archived   : {MLFLOW_MODEL_NAME} v{_ev.version} (was Staging)")

        if qg_pass:
            _mlflow_client.transition_model_version_stage(MLFLOW_MODEL_NAME, _new_version, "Staging")
            print(f"  Promoted   : {MLFLOW_MODEL_NAME} v{_new_version} -> Staging")
        else:
            print(f"  NOT promoted (quality gate FAIL: recall={qg_recall:.4f} < {qg_floor})")
    except Exception as _e:
        print(f"  [WARN] MLflow registration failed: {_e}")

    _thresh_json = {
        "category": CATEGORY,
        "active_threshold": float(opt_thresh),
        "recall_floor": qg_floor,
        "quality_gate_pass": qg_pass,
        "test_ap": _champ_ap,
        "mlflow_model_name": MLFLOW_MODEL_NAME,
        "mlflow_version": int(_new_version) if _new_version else None,
        "champion_name": CHAMPION_NAME,
        "model_uri": _model_uri,
        "flavor": "spark",
    }
    _tj_path = f"/tmp/threshold_final_{CATEGORY.lower()}.json"
    open(_tj_path, "w").write(_json_mod.dumps(_thresh_json, indent=2))
    try:
        _mlflow_client.log_artifact(_champ_run_id, _tj_path, "deployment")
    except Exception as _e:
        print(f"  [Warn] threshold artifact log failed: {_e}")

    try:
        _sm_client = boto3.client("sagemaker", region_name=REGION)
        try:
            _sm_client.describe_model_package_group(ModelPackageGroupName=MODEL_REGISTRY_NAME)
        except ClientError as _mpg_exc:
            if _mpg_exc.response["Error"]["Code"] not in ("ValidationException", "ResourceNotFound"):
                raise
            _sm_client.create_model_package_group(
                ModelPackageGroupName=MODEL_REGISTRY_NAME,
                ModelPackageGroupDescription=f"CUBIC MARS {CATEGORY} hardware OOS Spark ML",
            )
        _approve = "Approved" if qg_pass else "PendingManualApproval"
        _mp_resp = _sm_client.create_model_package(
            ModelPackageGroupName=MODEL_REGISTRY_NAME,
            ModelPackageDescription=(
                f"spark {CHAMPION_NAME} test_ap={_champ_ap:.4f} recall={qg_recall:.4f} "
                f"thresh={opt_thresh:.4f} QG={'PASS' if qg_pass else 'FAIL'}"
            ),
            ModelApprovalStatus=_approve,
            CustomerMetadataProperties={k: str(v) for k, v in _thresh_json.items() if v is not None},
        )
        _res["sm_model_package_arn"] = _mp_resp["ModelPackageArn"]
        print(f"  SageMaker metadata package: {_approve}")
    except Exception as _e:
        print(f"  [Warn] SageMaker metadata registration skipped: {_e}")
        _res["sm_model_package_arn"] = None

    _res.update({
        "mlflow_version": _new_version,
        "champion_run_id": _champ_run_id,
        "champion_name": MLFLOW_MODEL_NAME,
        "champion_ap": _champ_ap,
        "champion_artifact_path": "model",
        "champion_model": _champ_entry.get("model") or CHAMPION_MODEL,
        "champion_flavor": "spark",
        "qg_floor": qg_floor,
    })


In [ ]:
#  CELL 22 : Deploy SageMaker Endpoint (Spark champion -> native booster bundle)
# ---------------------------------------------------------------------------
# Gated behind PS1_DEPLOY_ENDPOINT (default FALSE) so that a Run All no longer
# creates billable infrastructure as a side effect. Deploying is opt-in, matching
# the PS5_PUBLISH convention already used elsewhere in this repo.
#
# Skipping this cell costs nothing downstream: CELL 24's cross-wired output is
# built from the predictions CELL 20 stores in memory, not from an endpoint.
#
#   To deploy:  os.environ["PS1_DEPLOY_ENDPOINT"] = "true"   then re-run this cell.
# ---------------------------------------------------------------------------
import os as _os

if _os.environ.get("PS1_DEPLOY_ENDPOINT", "false").strip().lower() not in ("1", "true", "yes", "y"):
    print("[SKIP] CELL 22 endpoint deploy - PS1_DEPLOY_ENDPOINT is not set.")
    print("       Set os.environ['PS1_DEPLOY_ENDPOINT'] = 'true' and re-run to deploy.")
    print("       CELL 24 does not require an endpoint; it uses CELL 20's predictions.")
else:
    # Trained with mlflow.spark; endpoint uses exported native booster + imputer medians.
    import os, tarfile, joblib, shutil as _shutil_sm, time
    import mlflow.spark
    import sagemaker.image_uris as sm_images
    from sagemaker.sklearn import SKLearnModel
    from botocore.exceptions import ClientError

    apply_ps1_resource_names(CATEGORIES)


    def _endpoint_status(endpoint_name):
        try:
            return SM_CLIENT.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        except ClientError as exc:
            if exc.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
                return None
            raise


    def _delete_endpoint_config_if_exists(config_name):
        try:
            SM_CLIENT.delete_endpoint_config(EndpointConfigName=config_name)
            print(f"  Deleted endpoint config: {config_name}")
        except ClientError as exc:
            if exc.response["Error"]["Code"] not in ("ValidationException", "ResourceNotFound"):
                raise


    def _wait_until_endpoint_gone(endpoint_name, timeout_s=900, poll_s=15):
        deadline = time.time() + timeout_s
        while time.time() < deadline:
            if _endpoint_status(endpoint_name) is None:
                return
            time.sleep(poll_s)
        raise TimeoutError(f"Endpoint {endpoint_name} still deleting after {timeout_s}s")


    def _prepare_endpoint_deploy(endpoint_name):
        status = _endpoint_status(endpoint_name)
        if status == "InService":
            print(f"  Endpoint {endpoint_name} already InService — skipping deploy")
            return False
        if status is not None:
            print(f"  Endpoint {endpoint_name} status={status} — deleting for redeploy ...")
            SM_CLIENT.delete_endpoint(EndpointName=endpoint_name)
            _wait_until_endpoint_gone(endpoint_name)
        _delete_endpoint_config_if_exists(endpoint_name)
        return True

    def _export_spark_champion_bundle(spark_pipeline, feature_cols, tag, out_dir):
        os.makedirs(out_dir, exist_ok=True)
        stages = spark_pipeline.stages
        imputer = stages[0]
        clf = stages[-1]
        medians = {}
        try:
            _sur = imputer.surrogateDF.collect()[0].asDict()
            for c in feature_cols:
                if c in _sur and _sur[c] is not None:
                    medians[c] = float(_sur[c])
        except Exception as _ie:
            print(f"  [WARN] imputer medians export: {_ie}")

        meta = {"feature_cols": list(feature_cols), "medians": medians, "classifier": type(clf).__name__}
        model_file = None

        if hasattr(clf, "get_booster"):
            model_file = os.path.join(out_dir, f"ps1_{tag}_xgb.json")
            clf.get_booster().save_model(model_file)
            meta["model_type"] = "xgboost"
            meta["model_file"] = os.path.basename(model_file)
        elif hasattr(clf, "getNativeModel"):
            model_file = os.path.join(out_dir, f"ps1_{tag}_lgb.txt")
            try:
                clf.getNativeModel().save_model(model_file)
                meta["model_type"] = "lightgbm"
                meta["model_file"] = os.path.basename(model_file)
            except Exception:
                model_file = None
        elif hasattr(clf, "saveNativeModel"):
            model_file = os.path.join(out_dir, f"ps1_{tag}_catboost.cbm")
            try:
                clf.saveNativeModel(model_file)
                meta["model_type"] = "catboost"
                meta["model_file"] = os.path.basename(model_file)
            except Exception:
                model_file = None

        if model_file is None:
            raise RuntimeError(
                f"Cannot export native booster for {type(clf).__name__} — "
                "deploy requires SparkXGB/LGB/CatBoost champion"
            )

        joblib.dump(meta, os.path.join(out_dir, f"ps1_{tag}_meta.joblib"))
        return meta

    _INF_TEMPLATE = 'import os, joblib\nimport numpy as np\nimport pandas as pd\n\ndef _impute(df, cols, medians):\n    out = df.copy()\n    for c in cols:\n        if c in out.columns and c in medians:\n            out[c] = out[c].astype(float).fillna(medians[c])\n    return out\n\ndef model_fn(model_dir):\n    meta = joblib.load(os.path.join(model_dir, "ps1_TAG_meta.joblib"))\n    thresh = joblib.load(os.path.join(model_dir, "ps1_TAG_threshold.joblib"))\n    bundle = {"meta": meta, "threshold": float(thresh), "model": None}\n    mtype = meta["model_type"]\n    mf = os.path.join(model_dir, meta["model_file"])\n    if mtype == "xgboost":\n        import xgboost as xgb\n        b = xgb.Booster()\n        b.load_model(mf)\n        bundle["model"] = b\n    elif mtype == "lightgbm":\n        import lightgbm as lgb\n        bundle["model"] = lgb.Booster(model_file=mf)\n    else:\n        from catboost import CatBoostClassifier\n        m = CatBoostClassifier()\n        m.load_model(mf)\n        bundle["model"] = m\n    return bundle\n\ndef _predict_proba(bundle, df):\n    meta = bundle["meta"]\n    cols = meta["feature_cols"]\n    X = _impute(df, cols, meta.get("medians", {})).reindex(columns=cols, fill_value=0.0).astype(float).values\n    mtype = meta["model_type"]\n    if mtype == "xgboost":\n        import xgboost as xgb\n        return bundle["model"].predict(xgb.DMatrix(X, feature_names=cols))\n    if mtype == "lightgbm":\n        return bundle["model"].predict(X)\n    return bundle["model"].predict_proba(X)[:, 1]\n\ndef predict_fn(input_data, bundle):\n    if not isinstance(input_data, pd.DataFrame):\n        input_data = pd.DataFrame(input_data, columns=bundle["meta"]["feature_cols"])\n    return _predict_proba(bundle, input_data).tolist()\n\ndef input_fn(request_body, content_type):\n    if content_type == "application/json":\n        import json\n        data = json.loads(request_body)\n        return pd.DataFrame([data]) if isinstance(data, dict) else pd.DataFrame(data)\n    if content_type == "text/csv":\n        from io import StringIO\n        return pd.read_csv(StringIO(request_body))\n    raise ValueError("Unsupported content type: " + str(content_type))\n\ndef output_fn(prediction, accept):\n    import json\n    return json.dumps(prediction), "application/json"\n'

    for CATEGORY in [DEVICE_CAT]:
        cat = CATEGORIES[CATEGORY]
        tag = CATEGORY.lower()
        _endpoint_name = cat["endpoint"]
        _res = CATEGORY_RESULTS[CATEGORY]
        opt_thresh = _res.get("opt_thresh", 0.5)

        CHAMPION_RUN_ID = _res.get("champion_run_id") or SPARK_ML_RESULTS.get(CHAMPION_NAME, {}).get("run_id")
        CHAMPION_AP = _res.get("champion_ap", SPARK_ML_RESULTS.get(CHAMPION_NAME, {}).get("test_ap", 0.0))
        MLFLOW_VERSION = _res.get("mlflow_version") or "local"

        spark_pipeline = _res.get("champion_model") or CHAMPION_MODEL
        if spark_pipeline is None and CHAMPION_RUN_ID:
            spark_pipeline = mlflow.spark.load_model(f"runs:/{CHAMPION_RUN_ID}/model")

        print(f"\n[{CATEGORY}] Exporting Spark champion '{CHAMPION_NAME}' for SageMaker endpoint ...")
        _code_dir = f"/tmp/sm_code_spark_{tag}"
        _shutil_sm.rmtree(_code_dir, ignore_errors=True)
        os.makedirs(_code_dir, exist_ok=True)

        _meta = _export_spark_champion_bundle(spark_pipeline, FEATURE_COLS, tag, _code_dir)
        joblib.dump(float(opt_thresh), os.path.join(_code_dir, f"ps1_{tag}_threshold.joblib"))

        _req = ["numpy", "pandas"]
        if _meta["model_type"] == "xgboost":
            _req.append("xgboost==2.1.3")
        elif _meta["model_type"] == "lightgbm":
            _req.append("lightgbm>=4.3")
        else:
            _req.append("catboost>=1.2")
        open(os.path.join(_code_dir, "requirements.txt"), "w").write("\n".join(_req) + "\n")
        open(os.path.join(_code_dir, "inference.py"), "w").write(_INF_TEMPLATE.replace("TAG", tag))

        with tarfile.open(f"/tmp/model_{tag}.tar.gz", "w:gz") as tar:
            for fn in os.listdir(_code_dir):
                tar.add(os.path.join(_code_dir, fn), arcname=fn)

        MODEL_S3_KEY = f"sagemaker/ps1-3d/{tag}/spark-model-v{MLFLOW_VERSION}/model.tar.gz"
        S3_CLIENT.upload_file(f"/tmp/model_{tag}.tar.gz", ARTIFACT_BUCKET, MODEL_S3_KEY)
        MODEL_DATA_URL = f"s3://{ARTIFACT_BUCKET}/{MODEL_S3_KEY}"
        print(f"  [{CATEGORY}] Uploaded bundle  model_type={_meta['model_type']}  AP={CHAMPION_AP:.4f}")

        _needs_deploy = _prepare_endpoint_deploy(_endpoint_name)

        if _needs_deploy:
            _sk_model = SKLearnModel(
                model_data=MODEL_DATA_URL,
                role=ROLE,
                entry_point="inference.py",
                source_dir=_code_dir,
                framework_version="1.2-1",
                env={"SAGEMAKER_MODEL_SERVER_TIMEOUT": "3600"},
                sagemaker_session=SM_SESSION,
            )
            print(f"  [{CATEGORY}] Deploying {_endpoint_name} (ml.m5.large) ...")
            _sk_model.deploy(
                initial_instance_count=1,
                instance_type="ml.m5.large",
                endpoint_name=_endpoint_name,
                wait=True,
            )
        _live_status = _endpoint_status(_endpoint_name)
        _res["endpoint_deployed"] = (_live_status == "InService")
        _res["deploy_model_type"] = _meta["model_type"]
        _res["deploy_threshold"] = opt_thresh
        if _live_status == "InService":
            print(f"  [{CATEGORY}] Endpoint ready: {_endpoint_name}")
        else:
            print(f"  [{CATEGORY}] Endpoint {_endpoint_name} status={_live_status!r} — deploy CELL 22 before CELL 23")

    print("\nSpark ML endpoint deployment complete.")


In [ ]:
#  CELL 23 : SageMaker Model Quality Monitor (Spark champion test scores)
from sagemaker.model_monitor import ModelQualityMonitor, CronExpressionGenerator, EndpointInput
from sagemaker.model_monitor.dataset_format import DatasetFormat
from botocore.exceptions import ClientError as _MQClientError

apply_ps1_resource_names(CATEGORIES)

def _mq_endpoint_status(name):
    try:
        return SM_CLIENT.describe_endpoint(EndpointName=name)["EndpointStatus"]
    except _MQClientError:
        return None


def _delete_mq_schedule_if_exists(schedule_name):
    try:
        SM_CLIENT.delete_monitoring_schedule(MonitoringScheduleName=schedule_name)
        print(f"  Deleted existing monitoring schedule: {schedule_name}")
    except _MQClientError as exc:
        code = exc.response.get("Error", {}).get("Code", "")
        if code not in ("ValidationException", "ResourceNotFound"):
            raise


MQ_MONITORS = {}
for CATEGORY in [DEVICE_CAT]:
    if CATEGORY not in CATEGORY_RESULTS:
        print(f"[{CATEGORY}] No CATEGORY_RESULTS — run CELL 13/20 first.")
        continue
    res = CATEGORY_RESULTS[CATEGORY]
    tag = CATEGORY.lower()
    cat = CATEGORIES[CATEGORY]
    _endpoint_name = cat["endpoint"]
    _ep_status = _mq_endpoint_status(_endpoint_name)
    if _ep_status != "InService":
        print(
            f"[{CATEGORY}] Endpoint '{_endpoint_name}' status={_ep_status!r} — "
            "Model Quality Monitor requires InService (run CELL 22 deploy first)."
        )
        res["endpoint_deployed"] = False
        continue
    res["endpoint_deployed"] = True
    _test_pdf = res.get("test_pdf")
    if _test_pdf is None or _test_pdf.empty:
        print(f"[{CATEGORY}] test_pdf missing — skipping monitor baseline")
        continue

    opt_thresh = float(res.get("opt_thresh", 0.5))
    y_prob_test = _test_pdf["score"].astype(float).values
    y_test = _test_pdf[TARGET_COL].astype(int).values
    y_pred_test = (y_prob_test >= opt_thresh).astype(int).astype(str)

    mq = ModelQualityMonitor(role=ROLE, instance_count=1, instance_type="ml.m5.large", sagemaker_session=SM_SESSION)
    mq_df = pd.DataFrame({"prediction": y_pred_test, "probability": y_prob_test, "label": y_test.astype(str)})
    mq_df.to_csv(f"/tmp/mq_baseline_{tag}.csv", index=False)
    S3_CLIENT.upload_file(
        f"/tmp/mq_baseline_{tag}.csv", ARTIFACT_BUCKET,
        f"sagemaker/ps1-3d/{tag}/mq-baseline/mq_baseline.csv",
    )

    import sagemaker as _sm_pkg
    if not hasattr(_sm_pkg, "Session"):
        from sagemaker.session import Session as _SMSession
        _sm_pkg.Session = _SMSession

    mq.suggest_baseline(
        baseline_dataset=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/mq-baseline/mq_baseline.csv",
        dataset_format=DatasetFormat.csv(header=True),
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/mq-baseline/output/",
        problem_type="BinaryClassification",
        inference_attribute="prediction",
        probability_attribute="probability",
        ground_truth_attribute="label",
        probability_threshold_attribute=str(opt_thresh),
        wait=True,
    )

    mq_endpoint_input = EndpointInput(
        endpoint_name=_endpoint_name,
        destination="/opt/ml/processing/input_data",
        inference_attribute="prediction",
        probability_attribute="probability",
        probability_threshold_attribute=opt_thresh,
    )
    if _mq_endpoint_status(_endpoint_name) != "InService":
        print(
            f"[{CATEGORY}] Endpoint '{_endpoint_name}' no longer InService after baseline — "
            "skipping monitoring schedule."
        )
        continue

    _schedule_name = f"chicago-ps1-3d-{tag}-model-quality"
    _delete_mq_schedule_if_exists(_schedule_name)
    try:
        mq.create_monitoring_schedule(
            monitor_schedule_name=_schedule_name,
            endpoint_input=mq_endpoint_input,
            ground_truth_input=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/ground-truth/",
            problem_type="BinaryClassification",
            output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps1-3d/{tag}/mq-output/",
            constraints=mq.suggested_constraints(),
            schedule_cron_expression=CronExpressionGenerator.daily(),
            enable_cloudwatch_metrics=True,
        )
    except _MQClientError as exc:
        _msg = exc.response.get("Error", {}).get("Message", str(exc))
        print(f"[{CATEGORY}] Model Quality Monitor schedule failed: {_msg}")
        continue
    print(f"[{CATEGORY}] Model Quality Monitor: {_schedule_name}")
    MQ_MONITORS[CATEGORY] = mq


In [ ]:
if _PS1_CONFIG_NON_DEFAULT:
    raise RuntimeError(
        "[EXPERIMENT] publishing is blocked because the label configuration is "
        "non-default. Cells 1-20 give the metrics; CELL 21 (MLflow registration) "
        "and CELL 24 (cross-wire S3 export) would publish an experimental label "
        "under the production name. Unset the PS1_* label vars to publish."
    )

#  CELL 24 : Cross-wired PS1 output at device + component/serial-ID level
# Expands device-level PS1 predictions to component/serial-ID grain so that
# dashboards and ServiceNow API calls can be made at BOTH device and component
# resolution.  Joins in:
#   â€¢ hw_config_current  → COMPONENT_SERIAL_NBR, COMPONENT_TYPE, COMPONENT_PART_NBR
#   • dim_device          → BUS_ID, bus_device_flag (mobile VALIDATOR bus mapping)
#   â€¢ device_ps2_chains  → cascade / coordinated-failure signals
#   â€¢ device_ps5_component → component-level MTTR / reliability tier
# Output: s3://{ARTIFACT_BUCKET}/chicago/device_ps1_cross_wired_daily/{gate|tvm|validator}/
# ServiceNow posting: one row per (DEVICE_KEY, COMPONENT_SERIAL_NBR, transit_day)
import json
S3_OPTS = {"client_kwargs": {"region_name": "us-east-1"}}
GOLD_PREFIX_KEY   = GOLD_BASE  # chicago/gold — defined in CELL 3
SILVER_PREFIX_KEY = SILVER_PREFIX  # chicago/silver — defined in CELL 3
#  1. Collect latest inference predictions from CATEGORY_RESULTS 
all_device_preds = []
for cat, res in CATEGORY_RESULTS.items():
    # CELL 20 stores 'predictions' in CATEGORY_RESULTS
    if "predictions" not in res or res["predictions"] is None:
        print(f"[{cat}] No predictions found — run CELL 20 (ranking) first.")
        continue
    pred_df = res["predictions"].copy()
    pred_df["device_category"] = cat
    # ensure required columns exist with safe defaults
    for col, default in [("ps1_fail_prob", 0.0), ("ps1_predicted", 0), ("threshold_used", 0.5)]:
        if col not in pred_df.columns:
            pred_df[col] = default
    all_device_preds.append(pred_df)
if not all_device_preds:
    print("No predictions available — skipping cross-wire output.")
else:
    device_preds_df = pd.concat(all_device_preds, ignore_index=True)
    if "DEVICE_KEY" not in device_preds_df.columns and "DEVICE_ID" in device_preds_df.columns:
        device_preds_df["DEVICE_KEY"] = device_preds_df["DEVICE_ID"].astype(str)
        print("[WARN] DEVICE_KEY missing in predictions — using DEVICE_ID as fallback.")
    #  2. Join component inventory 
    try:
        _hw_want = ["DEVICE_KEY", "COMPONENT_SERIAL_NBR", "COMPONENT_DESCRIPTION",
                    "FACILITY_ID", "OPERATOR_ID", "component_age_days"]
        hw_raw = pd.read_parquet(
            f"s3://{BUCKET}/{SILVER_PREFIX_KEY}/hw_config_current",
            storage_options=S3_OPTS,
        )
        _hw_cols = [c for c in _hw_want if c in hw_raw.columns]
        hw_df = hw_raw[_hw_cols].dropna(subset=["DEVICE_KEY"])
        if "COMPONENT_DESCRIPTION" in hw_df.columns:
            hw_df = hw_df.rename(columns={"COMPONENT_DESCRIPTION": "COMPONENT_TYPE"})
        comp_preds_df = device_preds_df.merge(hw_df, on="DEVICE_KEY", how="left")
        print(f"hw_config_current joined — {len(comp_preds_df):,} rows (device→component expand)")
    except Exception as e:
        print(f"hw_config_current join skipped: {e}")
        comp_preds_df = device_preds_df.copy()
    #  2b. Join dim_device for BUS_ID (mobile VALIDATOR bus mapping)
    try:
        _dd_want = ["DEVICE_ID", "DEVICE_KEY", "BUS_ID", "bus_device_flag",
                    "DEVICE_NAME", "FACILITY_NAME", "OPERATOR_ID"]
        dd_raw = pd.read_parquet(
            f"s3://{BUCKET}/{SILVER_PREFIX_KEY}/dim_device",
            storage_options=S3_OPTS,
        )
        if "is_current" in dd_raw.columns:
            dd_raw = dd_raw[dd_raw["is_current"] == True]  # noqa: E712
        _dd_cols = [c for c in _dd_want if c in dd_raw.columns]
        dd_df = dd_raw[_dd_cols].copy()
        _join_key = None
        if "DEVICE_KEY" in comp_preds_df.columns and "DEVICE_KEY" in dd_df.columns:
            _join_key = "DEVICE_KEY"
        elif "DEVICE_ID" in comp_preds_df.columns and "DEVICE_ID" in dd_df.columns:
            _join_key = "DEVICE_ID"
        if _join_key is None:
            raise ValueError("No DEVICE_KEY/DEVICE_ID overlap with dim_device")
        dd_df = dd_df.drop_duplicates(subset=[_join_key], keep="last")
        _overlap = [c for c in dd_df.columns if c != _join_key and c in comp_preds_df.columns]
        if _overlap:
            dd_df = dd_df.drop(columns=_overlap)
        for _drop in ("DEVICE_ID", "DEVICE_KEY"):
            if _drop != _join_key and _drop in dd_df.columns:
                dd_df = dd_df.drop(columns=[_drop])
        comp_preds_df = comp_preds_df.merge(dd_df, on=_join_key, how="left")
        _n_bus = int(comp_preds_df["BUS_ID"].notna().sum()) if "BUS_ID" in comp_preds_df.columns else 0
        _n_flag = int((comp_preds_df.get("bus_device_flag", pd.Series(dtype=bool)) == True).sum())  # noqa: E712
        print(
            f"dim_device joined on {_join_key} — "
            f"BUS_ID populated={_n_bus:,}/{len(comp_preds_df):,}  "
            f"bus_device_flag=True={_n_flag:,}"
        )
    except Exception as e:
        print(f"dim_device BUS_ID join skipped: {e}")
        if "BUS_ID" not in comp_preds_df.columns:
            comp_preds_df["BUS_ID"] = None
        if "bus_device_flag" not in comp_preds_df.columns:
            comp_preds_df["bus_device_flag"] = None
    #  3. Join PS2 cascade / coordinated-failure signals 
    try:
        ps2_df = pd.read_parquet(
            f"s3://{BUCKET}/{GOLD_PREFIX_KEY}/device_ps2_chains",
            storage_options=S3_OPTS
        )
        ps2_keep = ["DEVICE_KEY","transit_day",
                    "is_coordinated_station_failure","is_major_station_event",
                    "station_devices_failed","days_healthy_before_chain",
                    "no_prior_failure_in_window"]
        ps2_sig = (ps2_df[[c for c in ps2_keep if c in ps2_df.columns]]
                   .drop_duplicates(["DEVICE_KEY","transit_day"]))
        day_col = "transit_day" if "transit_day" in comp_preds_df.columns else "event_date"
        comp_preds_df = comp_preds_df.merge(
            ps2_sig.rename(columns={"transit_day": day_col}),
            on=["DEVICE_KEY", day_col], how="left"
        )
        print(f"PS2 cascade signals joined — coordinated failures: "
              f"{comp_preds_df.get('is_coordinated_station_failure', pd.Series()).sum()}")
    except Exception as e:
        print(f"PS2 join skipped: {e}")
    #  4. Join PS5 component reliability signals 
    try:
        ps5_df = pd.read_parquet(
            f"s3://{BUCKET}/{GOLD_PREFIX_KEY}/device_ps5_component",
            storage_options=S3_OPTS
        )
        ps5_keep = ["DEVICE_KEY","COMPONENT_SERIAL_NBR",
                    "ps5_risk_tier","avg_rolling_mttr_30d_min",
                    "total_failure_days_s28","last_failure_date_s28",
                    "avg_rolling_mttr_90d_min","max_downtime_ever_min"]
        ps5_sig = ps5_df[[c for c in ps5_keep if c in ps5_df.columns]].drop_duplicates(
            [c for c in ["DEVICE_KEY","COMPONENT_SERIAL_NBR"] if c in ps5_df.columns]
        )
        merge_keys = [k for k in ["DEVICE_KEY","COMPONENT_SERIAL_NBR"] if k in comp_preds_df.columns and k in ps5_sig.columns]
        comp_preds_df = comp_preds_df.merge(ps5_sig, on=merge_keys, how="left")
        print(f"PS5 component reliability joined on {merge_keys}")
    except Exception as e:
        print(f"PS5 join skipped: {e}")
    #  5. Anomaly flag — probability vs. device's 95th-percentile history 
    p95 = (device_preds_df.groupby("DEVICE_KEY")["ps1_fail_prob"]
           .quantile(0.95).rename("ps1_p95_hist"))
    comp_preds_df = comp_preds_df.merge(p95, on="DEVICE_KEY", how="left")
    comp_preds_df["is_prob_anomaly"] = (
        comp_preds_df["ps1_fail_prob"] > comp_preds_df["ps1_p95_hist"]
    ).fillna(False)
    #  6. Risk tier 
    def _risk_tier(p):
        if pd.isna(p):   return "UNKNOWN"
        if p >= 0.70:    return "CRITICAL"
        elif p >= 0.50:  return "HIGH"
        elif p >= 0.30:  return "MEDIUM"
        else:            return "LOW"
    comp_preds_df["ps1_risk_tier"] = comp_preds_df["ps1_fail_prob"].map(_risk_tier)
    #  7. SHAP top-3 features (global importances from CELL 19)
    for _shap_i in (1, 2, 3):
        comp_preds_df[f"shap_feat{_shap_i}"] = None
        comp_preds_df[f"shap_val{_shap_i}"] = np.nan
    for cat, res in CATEGORY_RESULTS.items():
        shap_top3 = res.get("shap_top3")
        if not shap_top3:
            continue
        _mask = comp_preds_df["device_category"] == cat
        for _shap_i, (_feat, _val) in enumerate(list(shap_top3.items())[:3], 1):
            comp_preds_df.loc[_mask, f"shap_feat{_shap_i}"] = _feat
            comp_preds_df.loc[_mask, f"shap_val{_shap_i}"] = float(_val)
    #  8. Write ServiceNow-ready parquet to S3 (GATE / TVM / VALIDATOR separate prefixes)
    day_col = "transit_day" if "transit_day" in comp_preds_df.columns else "event_date"
    ps1_xw_base = PS1_XW_S3
    out_paths = []
    for _cat in sorted(comp_preds_df["device_category"].dropna().unique()):
        _slug = str(_cat).strip().lower()
        _cat_df = comp_preds_df[comp_preds_df["device_category"] == _cat]
        _out = f"{ps1_xw_base}/{_slug}/"
        _cat_df.to_parquet(_out, index=False, storage_options=S3_OPTS)
        out_paths.append((_cat, _out, len(_cat_df)))
        print(f"  [{_cat}] {len(_cat_df):,} rows -> {_out}")
    #  9. Summary 
    print(f"\n{'='*70}")
    print(f"PS1 Cross-wired Output  ->  {ps1_xw_base}/{{gate|tvm|validator}}/")
    for _cat, _out, _n in out_paths:
        print(f"  {_cat}: {_n:,} rows at {_out}")
    print(f"{'='*70}")
    print(f"Rows (device Ã— component Ã— day) : {len(comp_preds_df):,}")
    unique_devices = comp_preds_df["DEVICE_KEY"].nunique()
    unique_comps   = comp_preds_df["COMPONENT_SERIAL_NBR"].nunique() if "COMPONENT_SERIAL_NBR" in comp_preds_df.columns else "N/A"
    print(f"Unique devices                  : {unique_devices:,}")
    print(f"Unique component serial IDs     : {unique_comps:,}" if isinstance(unique_comps,int) else f"Unique component serial IDs     : {unique_comps}")
    print(f"\nRisk tier distribution:")
    print(comp_preds_df["ps1_risk_tier"].value_counts().to_string())
    print(f"\nAnomaly devices (prob > p95)    : {comp_preds_df['is_prob_anomaly'].sum():,}")
    print(f"\nColumns in output ({len(comp_preds_df.columns)}):")
    col_groups = {
        "IDs"        : ["DEVICE_KEY","DEVICE_ID","BUS_ID","bus_device_flag","COMPONENT_SERIAL_NBR","COMPONENT_TYPE","FACILITY_ID","device_category",day_col],
        "PS1"        : ["ps1_fail_prob","ps1_predicted","threshold_used","ps1_risk_tier","is_prob_anomaly"],
        "PS2"        : ["is_coordinated_station_failure","is_major_station_event","station_devices_failed","days_healthy_before_chain"],
        "PS5"        : ["ps5_risk_tier","avg_rolling_mttr_30d_min","total_failure_days_s28","last_failure_date_s28"],
        "SHAP"       : ["shap_feat1","shap_val1","shap_feat2","shap_val2","shap_feat3","shap_val3"],
    }
    for grp, cols in col_groups.items():
        present = [c for c in cols if c in comp_preds_df.columns]
        print(f"  {grp:<12}: {present}")
    print(f"\nServiceNow API key: DEVICE_KEY → CI sys_id lookup via cmdb_ci_hardware.list")
    print(f"                    COMPONENT_SERIAL_NBR → serial_number on cmdb_ci_hardware")
